## Sobre este Trabalho

Este notebook implementa um **protótipo de assistente virtual aplicado à saúde da mulher**, com foco em triagem ginecológica e apoio à decisão clínica. O trabalho integra quatro componentes principais estudados na Fase 3:

1. **Fine-tuning com LoRA** (*Low-Rank Adaptation*): ajuste leve de um modelo de linguagem pré-treinado para o domínio de saúde da mulher, com dataset PubMedQA + sintético.
2. **RAG** (*Retrieval-Augmented Generation*): integração com base de conhecimento vetorial (FAISS) alimentada por literatura biomédica real (PubMedQA).
3. **LangGraph**: orquestração de **quatro fluxos** de triagem clínica — ginecológico, violência doméstica, obstétrico e prevenção.
4. **Avaliação quantitativa**: métricas ROUGE-1/2/L, cobertura de termos médicos e análise de bias/equidade demográfica.

> ⚠️ **Aviso importante:** Este sistema é um **protótipo acadêmico** e não deve ser utilizado em ambiente clínico real. Todas as orientações geradas são de apoio à decisão e não substituem a avaliação presencial por profissional habilitado.


## Célula 1 — Instalação de Dependências

**O que esta célula faz:**

Instala e verifica todas as bibliotecas necessárias para o projeto, garantindo versões compatíveis entre si.

**Por que esta ordem de instalação é importante:**

O ecossistema LangChain/LangGraph tem dependências interdependentes; instalar fora de ordem ou com versões conflitantes pode gerar erros difíceis de diagnosticar. Por isso:

1. Primeiro **desinstalamos** qualquer instalação anterior que possa gerar conflitos;
2. Depois instalamos `numpy` com versão controlada (necessário para compatibilidade com FAISS e PyTorch);
3. Em seguida instalamos o ecossistema LangChain completo (`langchain-core`, `langchain`, `langchain-community`, `langchain-huggingface`, `langchain-text-splitters`, `langchain-classic`) com versões mínimas definidas;
4. Instalamos `langgraph` separadamente (orquestrador de fluxos);
5. Instalamos as bibliotecas de ML: `transformers` (modelos de linguagem e tokenizers), `datasets` (manipulação de conjuntos de dados), `peft` (fine-tuning eficiente com LoRA), `trl` (treinamento com reforço), `accelerate` e `bitsandbytes` (otimizações de treino);
6. Instalamos ferramentas vetoriais: `sentence-transformers` (geração de embeddings) e `faiss-cpu` (busca vetorial eficiente); e
7. Instalamos utilitários complementares e verificamos todas as versões.

**Resultado esperado:** todas as bibliotecas instaladas e confirmadas sem erro, indicando que o ambiente está pronto para execução.


In [1]:
# ============================================================
# ETAPA 1 — INSTALAÇÃO DE DEPENDÊNCIAS
# Instala e verifica todas as bibliotecas do projeto.
# Ordem importa: numpy → langchain-core → langchain completo
#                → langgraph → ML (transformers/peft/trl)
#                → vetorial (sentence-transformers/faiss)
# ============================================================

# Passo 1: limpar instalações conflitantes anteriores
!pip uninstall -y langchain langchain-core langchain-community \
    langchain-huggingface langchain-text-splitters \
    langchain-classic langgraph langgraph-prebuilt 2>/dev/null

# Passo 2: numpy compatível
!pip install -q "numpy>=2.0.0,<2.5.0"

# Passo 3: ecossistema langchain alinhado
!pip install -q "langchain-core>=1.3.3"
!pip install -q "langchain>=1.0.0"
!pip install -q "langchain-community>=0.4.1"
!pip install -q "langchain-huggingface>=1.2.2"
!pip install -q "langchain-text-splitters>=1.1.2"
!pip install -q "langchain-classic"

# Passo 4: langgraph
!pip install -q "langgraph>=1.1.9"

!pip install requests==2.32.4

# Passo 5: ML / fine-tuning
!pip install -q "torchao>=0.16.0"
!pip install -q "transformers>=4.40.0"
!pip install -q "datasets>=2.18.0"
!pip install -q "peft>=0.10.0"
!pip install -q "trl>=0.8.0"
!pip install -q "accelerate>=0.27.0"
!pip install -q "bitsandbytes>=0.43.0"

# Passo 6: vetorial e embeddings
!pip install -q "sentence-transformers>=2.7.0"
!pip install -q "faiss-cpu>=1.8.0"

# Passo 7: utilitários
!pip install -q ijson cryptography huggingface_hub
!pip install -q rouge_score evaluate

# Passo 8: verificar versões instaladas
import importlib

pacotes = [
    ("numpy",                 "numpy"),
    ("langchain_core",        "langchain-core"),
    ("langchain_community",   "langchain-community"),
    ("langchain_huggingface", "langchain-huggingface"),
    ("langgraph",             "langgraph"),
    ("transformers",          "transformers"),
    ("peft",                  "peft"),
    ("sentence_transformers", "sentence-transformers"),
    ("torch",                 "torch"),
    ("faiss",                 "faiss-cpu"),
    ("ijson",                 "ijson"),
    ("datasets",              "datasets"),
]

print(f"\n{'Pacote':<35} {'Versão':<20} {'Status'}")
print("-" * 65)
erros = []
for modulo, nome in pacotes:
    try:
        mod = importlib.import_module(modulo)
        versao = getattr(mod, "__version__", "N/A")
        print(f"  {nome:<33} {versao:<20} OK")
    except Exception as e:
        print(f"  {nome:<33} {'ERRO':<20} FALHOU")
        erros.append((nome, str(e)))

print("-" * 65)
if erros:
    print(f"\nATENCAO: {len(erros)} modulo(s) com problema:")
    for nome, erro in erros:
        print(f"   ERRO {nome}: {erro}")
else:
    print("\nTodas as dependencias instaladas com sucesso!")

print("\nREINICIE O RUNTIME AGORA:")
print("Menu -> Runtime -> Restart session")

Found existing installation: langchain 1.3.1
Uninstalling langchain-1.3.1:
  Successfully uninstalled langchain-1.3.1
Found existing installation: langchain-core 1.4.0
Uninstalling langchain-core-1.4.0:
  Successfully uninstalled langchain-core-1.4.0
Found existing installation: langchain-community 0.4.2
Uninstalling langchain-community-0.4.2:
  Successfully uninstalled langchain-community-0.4.2
Found existing installation: langchain-huggingface 1.2.2
Uninstalling langchain-huggingface-1.2.2:
  Successfully uninstalled langchain-huggingface-1.2.2
Found existing installation: langchain-text-splitters 1.1.2
Uninstalling langchain-text-splitters-1.1.2:
  Successfully uninstalled langchain-text-splitters-1.1.2
Found existing installation: langchain-classic 1.0.7
Uninstalling langchain-classic-1.0.7:
  Successfully uninstalled langchain-classic-1.0.7
Found existing installation: langgraph 1.2.1
Uninstalling langgraph-1.2.1:
  Successfully uninstalled langgraph-1.2.1
Found existing installat

  peft                              0.19.1               OK
  sentence-transformers             5.4.1                OK
  torch                             2.10.0+cu128         OK
  faiss-cpu                         1.14.2               OK
  ijson                             3.5.0                OK
  datasets                          4.8.5                OK
-----------------------------------------------------------------

Todas as dependencias instaladas com sucesso!

REINICIE O RUNTIME AGORA:
Menu -> Runtime -> Restart session


## Célula 2 — Imports e Configuração Geral

**O que esta célula faz:**

Importa todas as bibliotecas necessárias, configura o sistema de logging para auditoria e define a semente aleatória para reprodutibilidade dos experimentos.

**Destaques técnicos:**

- **Logging estruturado:** O sistema de logging (`logging.basicConfig`) registra todas as interações do assistente com data/hora, nível de severidade e mensagem. Ele grava tanto no terminal (stream) quanto em um arquivo (`auditoria_assistente.log`), permitindo rastreabilidade completa — fundamental em aplicações de saúde. O logger recebe o nome `"AssistenteMedicoFeminino"` para identificação clara nos logs.
- **Semente aleatória (SEED = 42):** Garante que experimentos com aleatoriedade (como divisão de datasets e geração de texto) possam ser reproduzidos exatamente. Isso é essencial para comparar resultados entre diferentes execuções.
- **`warnings.filterwarnings("ignore")`:** Suprime avisos de bibliotecas que não afetam a execução, mas poluem a saída do notebook.
- **`TypedDict`:** Usado posteriormente para definir estruturas de estado tipadas no LangGraph, garantindo clareza sobre quais campos cada nodo do grafo deve preencher.


In [2]:
# ============================================================
# ETAPA 2 — IMPORTS E CONFIGURAÇÃO GERAL
# Importa bibliotecas, configura logging de auditoria e
# define semente aleatória para reprodutibilidade.
# ============================================================

import os
import json
import ijson
import hashlib
import logging
import datetime
import warnings
import random
import re
from pathlib import Path
from typing import TypedDict, Annotated, List, Optional, Dict, Any

import pandas as pd
import numpy as np
warnings.filterwarnings("ignore")

# Configuração de logging para auditoria (Etapa 4 - Segurança)
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler("auditoria_assistente.log")
    ]
)
logger = logging.getLogger("AssistenteMedicoFeminino")

# Semente para reprodutibilidade
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("Imports e configuracao geral concluidos.")
print(f"Data/Hora de inicio: {datetime.datetime.now().strftime('%d/%m/%Y %H:%M:%S')}")

Imports e configuracao geral concluidos.
Data/Hora de inicio: 23/05/2026 16:15:53


## Célula 3 — Carregamento do PubMedQA (ori_pqal.json)

**O que esta célula faz:**

Baixa o dataset **PubMedQA** diretamente da URL pública do GitHub, sem necessidade de Google Drive. O PubMedQA é um benchmark biomédico reconhecido contendo 1.000 perguntas clínicas extraídas de artigos do PubMed, cada uma com contextos (abstracts), resposta longa e decisão final (yes/no/maybe).

**Por que o PubMedQA é superior ao dataset de notícias:**
- Literatura biomédica revisada por pares, não texto jornalístico genérico;
- Cada registro tem PMID — fonte citável para o critério de *explainability* do edital;
- ~27% dos registros cobrem temas de saúde feminina diretamente relevantes.

**Transformações aplicadas:**
- `QUESTION` → `instrucao`; `LONG_ANSWER` → `resposta`; PMID → `fonte` citável;
- `final_decision` (yes/maybe/no) → `nivel_confianca` (0.90/0.72/0.55);
- Inferência de `categoria` por palavras-chave nos campos QUESTION + MESHES.

**Fallback:** se o download falhar, o notebook continua com o dataset sintético da Célula 4.


In [3]:
# ============================================================
# ETAPA 3 — CARREGAMENTO DO PUBMEDQA (ori_pqal.json)
#
# Substitui o news_dataset por literatura biomédica real.
# Download direto via URL pública — sem necessidade de Drive.
#
# Transformações:
#   QUESTION        → instrucao
#   LONG_ANSWER     → resposta
#   PMID (chave)    → fonte citável
#   final_decision  → nivel_confianca (yes=0.90 / maybe=0.72 / no=0.55)
#   MESHES + QUESTION → categoria (inferida por palavras-chave)
# ============================================================

import urllib.request
import json
import pandas as pd

URL_PUBMEDQA = (
    "https://raw.githubusercontent.com/pubmedqa/pubmedqa/"
    "refs/heads/master/data/ori_pqal.json"
)

PALAVRAS_CHAVE_SAUDE = [
    "gynecol", "obstetric", "pregnan", "menstrual", "breast cancer",
    "domestic violence", "contraceptiv", "menopause", "prenatal",
    "postpartum", "reproduct", "cervical", "uterine", "ovarian",
    "maternal", "endometri", "mammograph", "female", "woman", "women",
    "gender", "fertility", "eclampsia", "placenta", "lactation",
    "breastfeed", "ginecolog", "obstetr", "gravidez", "menopausa"
]

MAPA_DECISAO_CONFIANCA = {"yes": 0.90, "maybe": 0.72, "no": 0.55}

MAPA_CATEGORIA = {
    "breast":       "rastreamento_mama",
    "cervical":     "prevencao",
    "uterine":      "ginecologia",
    "ovarian":      "ginecologia",
    "endometri":    "ginecologia",
    "pregnan":      "obstetricia",
    "obstetric":    "obstetricia",
    "prenatal":     "obstetricia",
    "maternal":     "obstetricia",
    "postpartum":   "saude_mental_materna",
    "menstrual":    "ginecologia",
    "contraceptiv": "contracepcao",
    "menopause":    "climaterio",
    "reproduct":    "saude_reprodutiva",
    "violence":     "violencia_domestica",
    "lactation":    "saude_reprodutiva",
    "breastfeed":   "saude_reprodutiva",
}

def inferir_categoria(texto: str) -> str:
    t = texto.lower()
    for chave, cat in MAPA_CATEGORIA.items():
        if chave in t:
            return cat
    return "medicina_geral"

def contem_palavra_chave(texto: str) -> bool:
    t = texto.lower()
    return any(p in t for p in PALAVRAS_CHAVE_SAUDE)

def converter_registro(pmid: str, rec: dict) -> dict:
    questao  = rec.get("QUESTION", "")
    resposta = rec.get("LONG_ANSWER", "")
    decisao  = rec.get("final_decision", "maybe")
    meshes   = rec.get("MESHES", [])
    contextos = " ".join(rec.get("CONTEXTS", []))
    texto_cat = questao + " " + " ".join(meshes)
    return {
        "instrucao":       questao,
        "resposta":        resposta if resposta else contextos[:500],
        "categoria":       inferir_categoria(texto_cat),
        "fonte":           f"PubMed PMID:{pmid} | {', '.join(meshes[:3])}",
        "nivel_confianca": MAPA_DECISAO_CONFIANCA.get(decisao, 0.72),
        "contextos_raw":   contextos[:1500],
        "pmid":            pmid,
        "meshes":          meshes,
    }

# --- Download e parse ---
print(f"Baixando PubMedQA de:\n  {URL_PUBMEDQA}\n")
try:
    with urllib.request.urlopen(URL_PUBMEDQA, timeout=60) as resp:
        raw = json.loads(resp.read().decode("utf-8"))
    print(f"Download concluido. Total de registros: {len(raw):,}")
except Exception as e:
    print(f"Erro no download: {e}")
    print("Usando apenas dataset sintetico como fallback.")
    raw = {}

# --- Filtrar registros de saude feminina ---
registros_femininos = []
for pmid, rec in raw.items():
    texto = (
        rec.get("QUESTION", "") + " " +
        rec.get("LONG_ANSWER", "") + " " +
        " ".join(rec.get("MESHES", []))
    )
    if contem_palavra_chave(texto):
        registros_femininos.append(converter_registro(pmid, rec))

print(f"Registros filtrados (saude feminina): {len(registros_femininos)}")

if registros_femininos:
    df_noticias = pd.DataFrame(registros_femininos)
    print(f"\nShape do DataFrame: {df_noticias.shape}")
    print(f"\nDistribuicao por categoria:")
    print(df_noticias["categoria"].value_counts().to_string())
    display(df_noticias[["instrucao","categoria","nivel_confianca","fonte"]].head(5))
else:
    print("Nenhum registro filtrado. Dataset sintetico sera usado.")
    df_noticias = pd.DataFrame()


Baixando PubMedQA de:
  https://raw.githubusercontent.com/pubmedqa/pubmedqa/refs/heads/master/data/ori_pqal.json

Download concluido. Total de registros: 1,000
Registros filtrados (saude feminina): 787

Shape do DataFrame: (787, 8)

Distribuicao por categoria:
categoria
medicina_geral       639
obstetricia           55
rastreamento_mama     40
ginecologia           26
prevencao             20
climaterio             6
saude_reprodutiva      1


,instrucao,categoria,nivel_confianca,fonte
0,Landolt C and snellen e acuity: differences in...,medicina_geral,0.55,"PubMed PMID:16418930 | Adolescent, Adult, Aged"
1,Are the long-term results of the transanal pul...,medicina_geral,0.55,"PubMed PMID:17208539 | Child, Child, Preschool..."
2,Can tailored interventions increase mammograph...,medicina_geral,0.90,"PubMed PMID:10808977 | Cost-Benefit Analysis, ..."
3,Double balloon enteroscopy: is it efficacious ...,medicina_geral,0.90,PubMed PMID:23831910 | Community Health Center...
4,30-Day and 1-year mortality in emergency gener...,medicina_geral,0.72,"PubMed PMID:26037986 | Adult, Age Factors, Aged"


## Célula 4 — Dataset Sintético de Saúde da Mulher

**O que esta célula faz:**

Define um dataset sintético estruturado com 10 exemplos de pares pergunta–resposta cobrindo os principais temas de saúde da mulher: ginecologia, contracepção, obstetrícia, saúde mental materna, violência doméstica, prevenção/rastreamento e climatério.

**Estrutura de cada registro:**

Cada exemplo no dataset possui os seguintes campos:
- `instrucao`: a pergunta ou situação clínica apresentada ao assistente;
- `resposta`: a orientação especializada correspondente;
- `categoria`: categoria temática (ex.: `"ginecologia"`, `"obstetricia"`, `"violencia_domestica"`);
- `fonte`: referência ao protocolo ou diretriz que embasou a resposta (ex.: `"FEBRASGO 2023"`, `"INCA 2023"`); e
- `nivel_confianca`: score numérico indicando a confiança estimada na informação (0 a 1).

**Por que dados sintéticos:**

Em contextos de saúde, dados reais de pacientes são altamente sigilosos e sujeitos a regulações rigorosas (como a LGPD no Brasil). Dados sintéticos permitem desenvolver e testar pipelines de IA sem exposição de informações sensíveis, sendo prática comum em protótipos e pesquisas.

**Temas abordados e sua relevância:**

- **Ginecologia:** irregularidade menstrual e endometriose, condições de alta prevalência;
- **Contracepção:** efeitos colaterais de anticoncepcionais orais combinados;
- **Obstetrícia:** sinais de alerta na gestação (risco de pré-eclâmpsia, emergências obstétricas);
- **Saúde mental materna:** identificação de depressão pós-parto;
- **Violência doméstica:** protocolo de identificação e conduta — com notificação compulsória (SINAN) e acionamento de rede de proteção;
- **Prevenção:** rastreamento de câncer de colo do útero (Papanicolau) e câncer de mama (mamografia); e
- **Climatério:** manejo de sintomas da menopausa.

> **Nota sobre uso clínico:** As respostas do dataset são inspiradas em diretrizes nacionais e internacionais, mas **não reproduzem integralmente nenhum protocolo oficial**. Em um sistema real, a base de dados deve ser construída diretamente a partir de protocolos institucionais validados, devidamente versionados e atualizados.


In [4]:
# ============================================================
# ETAPA 4 — DATASET SINTÉTICO DE SAÚDE DA MULHER
#
# Define 10 exemplos de pares instrução-resposta cobrindo:
#   - Ginecologia (ciclo menstrual, endometriose)
#   - Contracepção (AOC, efeitos colaterais)
#   - Obstetrícia (sinais de alerta na gestação)
#   - Saúde mental materna (depressão pós-parto)
#   - Violência doméstica (protocolo de identificação)
#   - Prevenção (Papanicolau, mamografia)
#   - Climatério (menopausa, terapia hormonal)
#
# Cada registro inclui: instrucao, resposta, categoria,
# fonte (diretriz de referência) e nivel_confianca.
# ============================================================

DATASET_SAUDE_MULHER = [

    # --- Ginecologia e Ciclo Menstrual ---
    {
        "instrucao": "O que pode causar irregularidade no ciclo menstrual?",
        "resposta": (
            "Irregularidades no ciclo menstrual podem ser causadas por diversas condicoes, "
            "incluindo sindrome dos ovarios policisticos (SOP), disturbios tireoidianos, "
            "estresse intenso, variacoes significativas de peso, uso de anticoncepcionais, "
            "endometriose ou, em mulheres acima de 40 anos, inicio do climaterio. "
            "E fundamental realizar avaliacao clinica completa com ginecologista, incluindo "
            "dosagens hormonais (FSH, LH, TSH, prolactina) e ultrassonografia pelvica. "
            "AVISO: Esta informacao nao substitui consulta medica presencial."
        ),
        "categoria": "ginecologia",
        "fonte": "Protocolo FEBRASGO - Disturbios Menstruais 2023",
        "nivel_confianca": 0.92
    },
    {
        "instrucao": "Quais sao os sintomas da endometriose?",
        "resposta": (
            "A endometriose apresenta sintomas variaveis, sendo os principais: dismenorreia "
            "(colicas intensas antes e durante a menstruacao), dor pelvica cronica, dispareunia "
            "(dor durante relacao sexual), sangramento menstrual abundante ou irregular, dor ao "
            "evacuar ou urinar durante o periodo menstrual e, em alguns casos, infertilidade. "
            "O diagnostico definitivo e feito por videolaparoscopia. Encaminhe a paciente para "
            "avaliacao ginecologica especializada. AVISO: Nao diagnosticar definitivamente sem "
            "avaliacao clinica e exames complementares."
        ),
        "categoria": "ginecologia",
        "fonte": "Diretrizes FEBRASGO - Endometriose 2022",
        "nivel_confianca": 0.95
    },

    # --- Contracepcao ---
    {
        "instrucao": "Quais os efeitos colaterais mais comuns do anticoncepcional oral combinado?",
        "resposta": (
            "Os anticoncepcionais orais combinados (AOC) podem causar: nauseas (geralmente "
            "passageiras, melhoram tomando com alimento), cefaleia, spotting (sangramento "
            "intermenstrual nos primeiros meses), alteracoes de humor, diminuicao da libido, "
            "sensibilidade mamaria e alteracoes no fluxo menstrual. Efeitos raros mas graves "
            "incluem tromboembolismo venoso, especialmente em fumantes acima de 35 anos e "
            "pacientes com historico familiar. Contraindicacoes absolutas: enxaqueca com aura, "
            "hipertensao nao controlada, antecedente de trombose. Encaminhar para reavaliacao "
            "medica em caso de sintomas persistentes. Fonte: ANVISA / FDA / FEBRASGO."
        ),
        "categoria": "contracepcao",
        "fonte": "ANVISA - Bula Referencia + FEBRASGO 2023",
        "nivel_confianca": 0.93
    },

    # --- Pre-natal e Obstetrica ---
    {
        "instrucao": "Quais sao os sinais de alerta na gravidez que exigem atendimento imediato?",
        "resposta": (
            "SINAIS DE ALERTA NA GESTACAO: encaminhar imediatamente para servico de urgencia. "
            "1) Sangramento vaginal em qualquer trimestre. "
            "2) Dor abdominal intensa ou continua. "
            "3) Pressao arterial >= 140/90 mmHg (risco de pre-eclampsia). "
            "4) Edema subito de face, maos e pes. "
            "5) Cefaleia intensa e visao turva. "
            "6) Ausencia de movimentos fetais por mais de 12 horas. "
            "7) Febre acima de 38 graus C. "
            "8) Perda de liquido pela vagina (possivel rotura de membranas). "
            "PROTOCOLO: Acionar equipe obstetrica imediatamente. Nao aguardar proxima consulta."
        ),
        "categoria": "obstetricia",
        "fonte": "OMS - Cuidados no Pre-natal / FEBRASGO Emergencias Obstetricas 2023",
        "nivel_confianca": 0.98
    },
    {
        "instrucao": "Como identificar depressao pos-parto?",
        "resposta": (
            "A depressao pos-parto (DPP) afeta 10 a 15 porcento das puerpereas e pode se "
            "iniciar ate 12 meses apos o parto. Sinais de alerta: tristeza persistente por "
            "mais de 2 semanas, choro frequente sem causa aparente, sentimentos de inadequacao "
            "como mae, dificuldade de criar vinculo com o bebe, pensamentos de automutilacao "
            "ou de prejudicar o bebe (SINAL CRITICO: acionar psiquiatria imediatamente), "
            "insonia ou hipersonia alem do esperado, perda de apetite, isolamento social. "
            "Rastreamento recomendado: Escala de Edinburgh (EPDS) com ponto de corte >= 10. "
            "Encaminhar para equipe de saude mental perinatal. "
            "AVISO: Nao prescrever medicacao sem avaliacao psiquiatrica."
        ),
        "categoria": "saude_mental_materna",
        "fonte": "Literatura de Psiquiatria Perinatal / DSM-5 / Protocolo MS Brasil",
        "nivel_confianca": 0.94
    },

    # --- Violencia Domestica ---
    {
        "instrucao": "Quais sao os sinais que podem indicar violencia domestica em uma paciente?",
        "resposta": (
            "PROTOCOLO DE IDENTIFICACAO DE VIOLENCIA DOMESTICA (CONFIDENCIAL). "
            "Sinais fisicos: lesoes em diferentes estagios de cicatrizacao, hematomas em locais "
            "cobertos pela roupa, lesoes incompativeis com o mecanismo descrito, fraturas de defesa. "
            "Sinais comportamentais: acompanhante que responde pelas perguntas, paciente ansiosa "
            "ou com medo, minimizacao de lesoes, consultas frequentes por queixas vagas. "
            "Sinais ginecologicos: ISTs de repeticao, gestacoes nao planejadas frequentes, "
            "dor pelvica cronica sem causa organica identificada. "
            "ACAO OBRIGATORIA: Acolher em ambiente seguro e privado, sem a presenca do acompanhante. "
            "Notificacao compulsoria via Ficha SINAN. Acionar assistencia social e psicologia. "
            "Em risco imediato: acionar CREAS, Delegacia da Mulher (DDM) e, se necessario, SAMU. "
            "CONFIDENCIALIDADE: Dados registrados em prontuario restrito."
        ),
        "categoria": "violencia_domestica",
        "fonte": "Protocolo MS - Atencao as Mulheres em Situacao de Violencia / Lei Maria da Penha",
        "nivel_confianca": 0.97
    },

    # --- Prevencao / Rastreamento ---
    {
        "instrucao": "Qual a periodicidade recomendada para realizacao do exame Papanicolau?",
        "resposta": (
            "Conforme diretrizes do INCA e Ministerio da Saude: "
            "Inicio aos 25 anos para mulheres que ja tiveram relacao sexual. "
            "Periodicidade: anual nos dois primeiros exames consecutivos normais. "
            "Apos dois exames normais: a cada 3 anos. "
            "Mulheres imunossuprimidas (HIV positivo): anual indefinidamente. "
            "Encerramento: apos 64 anos com dois ultimos exames negativos nos ultimos 5 anos. "
            "Resultados alterados devem ser encaminhados para colposcopia. "
            "Fonte: INCA - Diretrizes Brasileiras para Rastreamento do Cancer do Colo do Utero 2023."
        ),
        "categoria": "prevencao",
        "fonte": "INCA - Diretrizes Rastreamento Colo do Utero 2023",
        "nivel_confianca": 0.96
    },
    {
        "instrucao": "Quais sao as recomendacoes de rastreamento para cancer de mama?",
        "resposta": (
            "Protocolo INCA e Ministerio da Saude Brasil: "
            "Mamografia de rastreamento: mulheres entre 50 e 69 anos, a cada 2 anos pelo SUS. "
            "Sociedade Brasileira de Mastologia: a partir dos 40 anos, anualmente. "
            "Risco elevado (BRCA1/2, historico familiar de primeiro grau): iniciar 10 anos antes "
            "do caso mais jovem na familia, nunca apos os 30 anos. "
            "Ultrassom mamario: complementar a mamografia em mamas densas. "
            "Sinais de alerta imediato: nodulo palpavel, inversao de mamilo, secrecao espontanea, "
            "alteracao de pele com aspecto de casca de laranja. Encaminhar para mastologista. "
            "Fonte: INCA + American Cancer Society adaptacao Brasil."
        ),
        "categoria": "prevencao",
        "fonte": "INCA - Controle do Cancer de Mama 2023 / American Cancer Society",
        "nivel_confianca": 0.96
    },

    # --- Climaterio ---
    {
        "instrucao": "Quais sao os sintomas do climaterio e como manejalos?",
        "resposta": (
            "O climaterio compreende o periodo de transicao do menacme para a menopausa, "
            "geralmente entre 40 e 65 anos. "
            "Sintomas vasomotores: fogachos e suores noturnos (mais comuns). "
            "Sintomas urogenitais: secura vaginal, dispareunia, urgencia urinaria. "
            "Sintomas psicologicos: irritabilidade, ansiedade, alteracoes de memoria e sono. "
            "Manifestacoes metabolicas: risco aumentado de osteoporose e doencas cardiovasculares. "
            "Manejo: Terapia Hormonal da Menopausa (THM) e a mais eficaz para sintomas vasomotores. "
            "Avaliar contraindicacoes: cancer de mama, trombose, cancer de endometrio. "
            "Alternativas nao hormonais: inibidores de recaptacao de serotonina, isoflavonas. "
            "AVISO: Individualizar conduta. Nao prescrever sem avaliacao especializada. "
            "Fonte: FEBRASGO - Consenso Climaterio 2023."
        ),
        "categoria": "climaterio",
        "fonte": "FEBRASGO - Consenso Climaterio e Menopausa 2023",
        "nivel_confianca": 0.91
    },

    # --- Saude Mental ---
    {
        "instrucao": "Como abordar uma paciente que demonstra sinais de ansiedade severa?",
        "resposta": (
            "Abordagem clinica para ansiedade severa em mulheres. "
            "1) ACOLHIMENTO: Escuta ativa, ambiente privado e seguro, linguagem empatica. "
            "2) AVALIACAO: Aplicar GAD-7 (Generalized Anxiety Disorder 7-item). "
            "3) RASTREAMENTO: Investigar gatilhos como violencia domestica, sobrecarga de "
            "cuidado, questoes reprodutivas, perdas gestacionais. "
            "4) ENCAMINHAMENTO: Para psicologia e/ou psiquiatria conforme severidade. "
            "5) SEGUIMENTO: Definir retorno breve em 1 a 2 semanas. "
            "AVISO: Nao prescrever benzodiazepinicos sem avaliacao psiquiatrica. Em gestantes, "
            "avaliar risco-beneficio criteriosamente. NUNCA minimizar queixas de ansiedade feminina."
        ),
        "categoria": "saude_mental",
        "fonte": "Protocolo MS - Saude Mental na APS / DSM-5",
        "nivel_confianca": 0.90
    },
]

df_sintetico = pd.DataFrame(DATASET_SAUDE_MULHER)

# Combinar dataset sintetico (PT/FEBRASGO) + PubMedQA filtrado
if not df_noticias.empty:
    colunas_comuns = ["instrucao", "resposta", "categoria", "fonte", "nivel_confianca"]
    df_pubmed = df_noticias[colunas_comuns].copy()
    df_treino = pd.concat([df_sintetico, df_pubmed], ignore_index=True)
    print(f"Dataset combinado:")
    print(f"   Sintetico (PT/FEBRASGO) : {len(df_sintetico)} registros")
    print(f"   PubMedQA (EN/PubMed)    : {len(df_pubmed)} registros")
    print(f"   TOTAL                   : {len(df_treino)} registros")
else:
    df_treino = df_sintetico
    print(f"Dataset sintetico apenas: {len(df_treino)} registros")

print(f"\nDistribuicao por categoria:")
print(df_treino["categoria"].value_counts().to_string())


Dataset combinado:
   Sintetico (PT/FEBRASGO) : 10 registros
   PubMedQA (EN/PubMed)    : 787 registros
   TOTAL                   : 797 registros

Distribuicao por categoria:
categoria
medicina_geral          639
obstetricia              56
rastreamento_mama        40
ginecologia              28
prevencao                22
climaterio                7
contracepcao              1
saude_mental_materna      1
violencia_domestica       1
saude_mental              1
saude_reprodutiva         1


## Célula 5 — Pré-processamento e Anonimização

**O que esta célula faz:**

Implementa a classe `PreProcessadorMedicoFeminino`, responsável por preparar os dados para o fine-tuning. A classe aplica três operações principais: **anonimização**, **normalização** e **formatação para fine-tuning**.

**Anonimização (`anonimizar`):**

Remove dados pessoais identificáveis usando expressões regulares (*regex*). Os padrões detectados e substituídos incluem:
- CPF (`000.000.000-00` → `[CPF_ANONIMIZADO]`);
- Datas de nascimento (`dd/mm/yyyy` → `[DATA_NASCIMENTO]`);
- CEP (`00000-000` → `[CEP]`);
- Telefones → `[TELEFONE]`;
- E-mails → `[EMAIL]`;
- Números de prontuário → `[PRONTUARIO_ANONIMIZADO]`; e
- IDs de 7 dígitos → `[ID_PACIENTE]`.

Essa etapa simula o cumprimento da **LGPD (Lei Geral de Proteção de Dados)** em sistemas de saúde.

**Normalização terminológica (`normalizar_texto`):**

Padroniza termos leigos para terminologia médica (por exemplo: `"gravida"` → `"gestante"`, `"dor de barriga"` → `"dor abdominal"`). Isso melhora a coerência do dataset e aproxima o vocabulário ao que seria encontrado em protocolos clínicos.

**Formatação para fine-tuning (`formatar_para_fine_tuning`):**

Aplica o template **Alpaca**, amplamente usado para fine-tuning de LLMs orientados a instruções (LLaMA, Falcon etc.). O template estrutura cada exemplo como:
```
### Instrucao:
Você é um assistente médico especializado em saúde da mulher...
{instrucao}

### Resposta:
{resposta}
```

**Pipeline completo (`processar_dataframe`):**

O método aplica sequencialmente: anonimização → normalização → formatação, e adiciona:
- `hash_anonimizacao`: hash SHA-256 dos 12 primeiros caracteres da instrução processada, para rastreabilidade sem expor o conteúdo; e
- `timestamp_processamento`: registro de data/hora do processamento.

**Resultado:** O DataFrame `df_processado` contém a coluna `texto_fine_tuning` pronta para alimentar o processo de treinamento do modelo na próxima célula.


In [7]:
# ============================================================
# ETAPA 5 — PRÉ-PROCESSAMENTO E ANONIMIZAÇÃO
#
# Classe PreProcessadorMedicoFeminino: pipeline completo de
# preparação dos dados para fine-tuning.
#
# Métodos principais:
#   anonimizar()              → remove CPF, datas, CEP, emails etc.
#   normalizar_texto()        → padroniza termos leigos para médicos
#   formatar_para_fine_tuning()→ aplica template Alpaca
#   processar_dataframe()     → pipeline completo no DataFrame
#
# Adiciona colunas: instrucao_processada, resposta_processada,
# texto_fine_tuning, hash_anonimizacao, timestamp_processamento
# ============================================================


import unicodedata

class PreProcessadorMedicoFeminino:
    """
    Pre-processador especializado para dados medicos de saude da mulher.
    Implementa anonimizacao, normalizacao e formatacao para fine-tuning.
    """

    PADROES_ANONIMIZACAO = {
        r'\b\d{3}\.\d{3}\.\d{3}-\d{2}\b'  : '[CPF_ANONIMIZADO]',
        r'\b\d{1,2}/\d{1,2}/\d{4}\b'  : '[DATA_NASCIMENTO]',
        r'\b\d{5}-\d{3}\b'  : '[CEP]',
        r'\b(?:\(?\d{2}\)?\s?)?\d{4,5}-\d{4}\b' : '[TELEFONE]',
        r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}' : '[EMAIL]',
        r'prontuario\s*n[°º]?\s*\d+'  : '[PRONTUARIO_ANONIMIZADO]',
        r'\b\d{7}\b'  : '[ID_PACIENTE]',
    }

    NORMALIZACAO_TERMOS = {
        "remedios"             : "medicamentos",
        "remedio"              : "medicamento",
        "dor de barriga"       : "dor abdominal",
        "dor nas costas"       : "lombalgia",
        "enjoo na gravidez"    : "emese gravidica",
        "menstruacao atrasada" : "atraso menstrual",
        "gravida"              : "gestante",
    }

    def anonimizar(self, texto: str) -> str:
        """Remove dados pessoais identificaveis do texto."""
        for padrao, substituto in self.PADROES_ANONIMIZACAO.items():
            texto = re.sub(padrao, substituto, texto, flags=re.IGNORECASE)
        return texto

    def normalizar_texto(self, texto: str) -> str:
        """Normaliza terminologia e encoding."""
        texto = unicodedata.normalize("NFC", texto)
        texto = re.sub(r"\s+", " ", texto).strip()
        for termo_leigo, termo_medico in self.NORMALIZACAO_TERMOS.items():
            texto = re.sub(
                r'\b' + re.escape(termo_leigo) + r'\b',
                termo_medico,
                texto,
                flags=re.IGNORECASE
            )
        return texto

    def formatar_para_fine_tuning(self, instrucao: str, resposta: str) -> str:
        """
        Formata o par instrucao-resposta no template Alpaca,
        compativel com fine-tuning de LLMs (LLaMA, Falcon etc.)
        """
        template = (
            "### Instrucao:\n"
            "Voce e um assistente medico especializado em saude da mulher. "
            "Responda de forma precisa, empatica e sempre recomende avaliacao "
            "presencial quando necessario.\n\n"
            f"{instrucao}\n\n"
            "### Resposta:\n"
            f"{resposta}"
        )
        return template

    def processar_dataframe(self, df: pd.DataFrame) -> pd.DataFrame:
        """Aplica todo o pipeline de pre-processamento ao DataFrame."""
        df = df.copy()
        df["instrucao_processada"] = df["instrucao"].apply(
            lambda x: self.normalizar_texto(self.anonimizar(x))
        )
        df["resposta_processada"] = df["resposta"].apply(
            lambda x: self.normalizar_texto(self.anonimizar(x))
        )
        df["texto_fine_tuning"] = df.apply(
            lambda row: self.formatar_para_fine_tuning(
                row["instrucao_processada"],
                row["resposta_processada"]
            ),
            axis=1
        )
        df["hash_anonimizacao"] = df["instrucao_processada"].apply(
            lambda x: hashlib.sha256(x.encode()).hexdigest()[:12]
        )
        df["timestamp_processamento"] = datetime.datetime.now().isoformat()
        return df


texto_teste = """
Paciente Maria, CPF 123.456.789-00, telefone (61) 99999-8888,
email maria@email.com, data 20/05/2026, CEP 72400-000.
"""

preprocessador = PreProcessadorMedicoFeminino()
df_processado  = preprocessador.processar_dataframe(df_treino)

print("Pre-processamento concluido!")
print(f"\nExemplo de texto formatado para fine-tuning:")
print("=" * 70)
print(df_processado["texto_fine_tuning"].iloc[0])
print("=" * 70)
print(f"\nHashes de anonimizacao gerados:")
print(df_processado["hash_anonimizacao"].tolist())
processador = PreProcessadorMedicoFeminino()
print(processador.anonimizar(texto_teste))

Pre-processamento concluido!

Exemplo de texto formatado para fine-tuning:
### Instrucao:
Voce e um assistente medico especializado em saude da mulher. Responda de forma precisa, empatica e sempre recomende avaliacao presencial quando necessario.

O que pode causar irregularidade no ciclo menstrual?

### Resposta:
Irregularidades no ciclo menstrual podem ser causadas por diversas condicoes, incluindo sindrome dos ovarios policisticos (SOP), disturbios tireoidianos, estresse intenso, variacoes significativas de peso, uso de anticoncepcionais, endometriose ou, em mulheres acima de 40 anos, inicio do climaterio. E fundamental realizar avaliacao clinica completa com ginecologista, incluindo dosagens hormonais (FSH, LH, TSH, prolactina) e ultrassonografia pelvica. AVISO: Esta informacao nao substitui consulta medica presencial.

Hashes de anonimizacao gerados:
['8593314287fc', '2a320db501fe', '7523ee2ddcde', 'a8de8931dcc8', '2667371dbb1c', 'c02271111229', '83726e34e4fc', '7e3880f909cc', 'e9

## Célula 6 — Simulação de Fine-tuning com PEFT/LoRA

**O que esta célula faz:**

Realiza o ajuste fino (*fine-tuning*) do modelo de linguagem GPT-2 utilizando a técnica **LoRA** (*Low-Rank Adaptation*), que permite treinar apenas uma fração dos parâmetros do modelo, tornando o processo viável mesmo em ambientes com recursos computacionais limitados como o Google Colab.

**Por que LoRA (Low-Rank Adaptation)?**

Modelos de linguagem modernos possuem bilhões de parâmetros. Retreiná-los completamente demanda recursos computacionais enormes (múltiplas GPUs, dias de processamento). O LoRA resolve isso adicionando **matrizes de baixo posto** (*low-rank matrices*) acopladas a camadas específicas do modelo. Apenas essas matrizes são treinadas, representando uma fração minúscula do total de parâmetros — sem perda significativa de qualidade para tarefas de domínio.

**Configuração LoRA utilizada:**
- `r=8`: posto das matrizes adicionadas (controla capacidade vs. eficiência);
- `lora_alpha=32`: fator de escala dos pesos LoRA;
- `target_modules=["c_attn"]`: camada de atenção do GPT-2 onde o LoRA é aplicado (em modelos como LLaMA seriam `["q_proj", "v_proj"]`);
- `lora_dropout=0.05`: regularização para evitar overfitting; e
- `task_type=TaskType.CAUSAL_LM`: tipo de tarefa — modelagem de linguagem causal (geração de texto).

**Modelo base: GPT-2**

Neste notebook de demonstração, usa-se o `gpt2` (modelo padrão do Hugging Face). O GPT-2 é adequado para demonstrar o pipeline completo de fine-tuning em ambiente Colab, pois é leve o suficiente para rodar em CPU/GPU T4. Em produção, recomenda-se modelos maiores como `meta-llama/Llama-2-7b-hf` ou `tiiuae/falcon-7b`, simplesmente substituindo o nome do modelo.

**Preparação do dataset:**

O dataset processado é convertido para o formato `Dataset` do Hugging Face e tokenizado com comprimento máximo de 512 tokens e padding. Os `labels` são cópias dos `input_ids` — padrão para modelagem de linguagem causal (o modelo aprende a prever o próximo token).

**Configuração de treinamento:**
- 3 épocas de treinamento;
- Batch size de 2 (limitação de memória do Colab);
- Avaliação e salvamento a cada época (`eval_strategy="epoch"`, `save_strategy="epoch"`);
- Seleção do melhor modelo ao final (`load_best_model_at_end=True`); e
- Sem uso de mixed-precision (`fp16=False`) para compatibilidade com CPU.

**Resultado:** O modelo ajustado é salvo em `./modelo_saude_mulher_final`, juntamente com o tokenizer. Este modelo será usado nas próximas células para geração de respostas.


In [8]:
# ============================================================
# ETAPA 6 — FINE-TUNING COM PEFT/LoRA  (versao otimizada Colab)
#
# PROBLEMA ORIGINAL — por que demorava 2 dias:
#   max_length=512  : cada amostra ocupa 512 tokens mesmo que a resposta
#                     tenha 150 tokens uteis → 3x mais computo desnecessario
#   batch_size=2    : poucos samples por passo → muitos passos de gradient
#   fp16=False      : float32 na GPU T4 é 2x mais lento que float16
#   epochs=3        : 3 passagens pelo dataset, sendo que 1 ja e suficiente
#                     para datasets pequenos (<300 amostras)
#   sem max_steps   : sem teto de tempo — dataset grande → treino infinito
#   save por epoca  : I/O de checkpoint a cada epoca adiciona minutos
#
# OTIMIZACOES APLICADAS (sem perda de aprendizado relevante):
#   1. max_length 512 → 256        reduz tokens/amostra pela metade
#   2. MAX_AMOSTRAS_TREINO = 80    limita dataset (10 sinteticos + 70 PubMedQA)
#   3. batch_size 2 → 4            mais throughput por passo
#   4. gradient_accumulation = 4   simula batch=16 sem estourar VRAM
#   5. fp16 = USE_GPU              float16 so quando ha GPU (CPU nao suporta)
#   6. epochs 3 → 1                1 epoca suficiente para dominio pequeno
#   7. max_steps = 80              teto absoluto de seguranca
#   8. group_by_length = True      agrupa amostras similares, menos padding
#   9. dataloader_num_workers = 2  tokenizacao paralela
#  10. save_strategy = "no"        elimina I/O de checkpoints intermediarios
#
# TEMPO ESTIMADO APOS OTIMIZACOES:
#   GPU T4 (Colab gratis) : ~5-10 minutos
#   CPU only              : ~20-30 minutos
# ============================================================

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from datasets import Dataset
from peft import LoraConfig, get_peft_model, TaskType
import torch
import time

# ── Detectar hardware disponivel ──────────────────────────────
USE_GPU = torch.cuda.is_available()
DEVICE  = "cuda" if USE_GPU else "cpu"
print(f"Hardware detectado : {DEVICE.upper()}")
if USE_GPU:
    nome_gpu = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU               : {nome_gpu}  ({vram_gb:.1f} GB VRAM)")
    print("fp16              : ATIVADO (2x velocidade na T4)")
else:
    print("Rodando em CPU — tempo estimado: ~20-30 min com as otimizacoes")
    print("Para acelerar: Runtime -> Change runtime type -> T4 GPU")

# ── Limitar amostras para controlar tempo ─────────────────────
# 10 sinteticos + ate 70 do PubMedQA = 80 amostras maximo.
# Suficiente para demonstrar fine-tuning sem explodir o tempo.
MAX_AMOSTRAS_TREINO = 80

df_treino_limitado = df_processado.head(MAX_AMOSTRAS_TREINO).reset_index(drop=True)
n_sinteticos = min(10, len(df_treino_limitado))
n_pubmed     = max(0, len(df_treino_limitado) - 10)
print(f"\nAmostras usadas no treino : {len(df_treino_limitado)} "
      f"(de {len(df_processado)} disponiveis)")
print(f"  Sinteticas (PT/FEBRASGO) : {n_sinteticos}")
print(f"  PubMedQA (EN/PubMed)     : {n_pubmed}")

# ── Configuracao LoRA (mantida intacta) ───────────────────────
LORA_CONFIG = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=32,
    target_modules=["c_attn"],   # GPT-2; LLaMA usaria ["q_proj","v_proj"]
    lora_dropout=0.05,
    bias="none",
    inference_mode=False
)

MODELO_BASE = "gpt2"
print(f"\nCarregando modelo base : {MODELO_BASE}")

tokenizer = AutoTokenizer.from_pretrained(MODELO_BASE)
tokenizer.pad_token = tokenizer.eos_token

# float16 na GPU (2x mais rapido), float32 na CPU (unico suportado)
modelo = AutoModelForCausalLM.from_pretrained(
    MODELO_BASE,
    torch_dtype=torch.float16 if USE_GPU else torch.float32,
)

modelo_lora = get_peft_model(modelo, LORA_CONFIG)
print("\nParametros treinaveis vs total:")
modelo_lora.print_trainable_parameters()

# ── Tokenizacao com max_length reduzido ───────────────────────
# 256 tokens cabem ~200 palavras — suficiente para respostas medicas
# do template Alpaca usado neste projeto.
MAX_LENGTH = 256  # era 512

def tokenizar(exemplo):
    tokens = tokenizer(
        exemplo["texto_fine_tuning"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length"
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

dataset_hf = Dataset.from_pandas(
    df_treino_limitado[["texto_fine_tuning"]].reset_index(drop=True)
)
dataset_tokenizado = dataset_hf.map(tokenizar, batched=True)
dataset_split      = dataset_tokenizado.train_test_split(test_size=0.2, seed=SEED)

n_treino = len(dataset_split["train"])
n_val    = len(dataset_split["test"])
print(f"\nSplit do dataset:")
print(f"   Treino    : {n_treino} amostras")
print(f"   Validacao : {n_val} amostras")

# ── Calcular estimativa de tempo ──────────────────────────────
BATCH_SIZE = 4
GRAD_ACCUM = 4
MAX_STEPS  = 80  # teto absoluto de seguranca

steps_estimados = max(1, n_treino // (BATCH_SIZE * GRAD_ACCUM))
tempo_gpu_min   = steps_estimados * 4  / 60
tempo_cpu_min   = steps_estimados * 20 / 60
print(f"\nPassos estimados    : {steps_estimados} (teto: {MAX_STEPS})")
print(f"Tempo estimado GPU  : ~{tempo_gpu_min:.0f} min")
print(f"Tempo estimado CPU  : ~{tempo_cpu_min:.0f} min")

# ── Argumentos de treinamento otimizados ──────────────────────
args_treinamento = TrainingArguments(
    output_dir                  = "./modelo_saude_mulher",
    num_train_epochs            = 1,        # era 3
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM,
    max_steps                   = MAX_STEPS,
    warmup_steps                = 5,        # era 10
    weight_decay                = 0.01,
    logging_dir                 = "./logs_treinamento",
    logging_steps               = 5,
    eval_strategy               = "steps",  # era "epoch"
    eval_steps                  = 20,
    save_strategy               = "no",     # era "epoch" — elimina I/O
    fp16                        = USE_GPU,  # era False — ativa apenas na GPU
    dataloader_num_workers      = 2,        # novo — tokenizacao paralela
    group_by_length             = True,     # novo — menos padding
    report_to                   = "none",
    seed                        = SEED,
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

trainer = Trainer(
    model         = modelo_lora,
    args          = args_treinamento,
    train_dataset = dataset_split["train"],
    eval_dataset  = dataset_split["test"],
    data_collator = data_collator,
)

# ── Treinar com cronometro ─────────────────────────────────────
print("\n" + "=" * 55)
print("Iniciando fine-tuning otimizado...")
print("=" * 55)
t0 = time.time()
trainer.train()
t1 = time.time()
duracao_min = (t1 - t0) / 60
print(f"\nFine-tuning concluido em {duracao_min:.1f} minutos")

# ── Salvar modelo ─────────────────────────────────────────────
modelo_lora.save_pretrained("./modelo_saude_mulher_final")
tokenizer.save_pretrained("./modelo_saude_mulher_final")
print("Modelo salvo em './modelo_saude_mulher_final'")

# ── Resumo final ──────────────────────────────────────────────
print("\n" + "=" * 55)
print("RESUMO DO TREINAMENTO")
print("=" * 55)
historico_loss = trainer.state.log_history
losses = [e["loss"] for e in historico_loss if "loss" in e and "eval_loss" not in e]
if losses:
    print(f"  Loss inicial  : {losses[0]:.4f}")
    print(f"  Loss final    : {losses[-1]:.4f}")
    print(f"  Reducao       : {(1 - losses[-1]/losses[0])*100:.1f}%")
print(f"  Duracao total : {duracao_min:.1f} min")
print(f"  Device        : {DEVICE.upper()}")
print(f"  Amostras      : {n_treino} treino / {n_val} validacao")


Hardware detectado : CUDA
GPU               : Tesla T4  (15.6 GB VRAM)
fp16              : ATIVADO (2x velocidade na T4)

Amostras usadas no treino : 80 (de 797 disponiveis)
  Sinteticas (PT/FEBRASGO) : 10
  PubMedQA (EN/PubMed)     : 70

Carregando modelo base : gpt2


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Parametros treinaveis vs total:
trainable params: 294,912 || all params: 124,734,720 || trainable%: 0.2364


...

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.



Iniciando fine-tuning otimizado...


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss,Validation Loss
20,5.033379,4.719486
40,4.974460,4.659143
60,4.884864,4.583949
80,4.871409,4.550124



Fine-tuning concluido em 0.6 minutos
Modelo salvo em './modelo_saude_mulher_final'

RESUMO DO TREINAMENTO
  Loss inicial  : 5.0985
  Loss final    : 4.8714
  Reducao       : 4.5%
  Duracao total : 0.6 min
  Device        : CUDA
  Amostras      : 64 treino / 16 validacao


## Célula 6b — Avaliação Quantitativa do Modelo Fine-tuned

**O que esta célula faz:**

Preenche a lacuna de métricas formais apontada na avaliação. Calcula três dimensões de avaliação:

1. **ROUGE-1, ROUGE-2, ROUGE-L**: métricas padrão de sobreposição de n-gramas entre as respostas geradas e as respostas de referência. Amplamente usadas em avaliação de LLMs para sumarização e QA.
2. **Cobertura de termos médicos**: métrica de domínio que mede a proporção de termos clínicos especializados (ginecologia, obstetrícia, menopausa etc.) presentes nas respostas geradas — indicador de especialização no domínio.
3. **Análise de bias/equidade demográfica**: avalia se o modelo responde de forma equitativa a perguntas formuladas para diferentes perfis de pacientes (jovem/adulta/idosa, baixa/alta renda, rural/urbana, diferentes etnias). Detecta disparidades sistemáticas nas respostas.

**Por que isso importa para o edital:**
O edital exige explicitamente métricas de avaliação especializada e análise de bias e equidade entre grupos demográficos.


In [9]:
# ============================================================
# ETAPA 6b — AVALIAÇÃO QUANTITATIVA DO MODELO FINE-TUNED
#
# 1. ROUGE-1/2/L: qualidade textual das respostas geradas
# 2. Cobertura de termos médicos: especialização no domínio
# 3. Análise de bias demográfico: equidade entre perfis de pacientes
#
# Referência: rouge_score + evaluate (já instalados na Etapa 1)
# ============================================================

from evaluate import load
import numpy as np
import pandas as pd
import torch

rouge = load("rouge")

# ============================================================
# PARTE 1 — MÉTRICAS DE TREINAMENTO (loss)
# ============================================================
print("=" * 65)
print("PARTE 1 — HISTÓRICO DE LOSS DO TREINAMENTO")
print("=" * 65)

historico = trainer.state.log_history
losses_treino = [e["loss"]      for e in historico if "loss"      in e and "eval_loss" not in e]
losses_eval   = [e["eval_loss"] for e in historico if "eval_loss" in e]

if losses_treino:
    reducao_pct = (1 - losses_treino[-1] / losses_treino[0]) * 100
    print(f"  Loss inicial  (treino)    : {losses_treino[0]:.4f}")
    print(f"  Loss final    (treino)    : {losses_treino[-1]:.4f}")
    print(f"  Reducao de loss           : {reducao_pct:.1f}%")
if losses_eval:
    print(f"  Loss validacao (melhor)   : {min(losses_eval):.4f}")
    print(f"  Loss validacao (final)    : {losses_eval[-1]:.4f}")

# ============================================================
# PARTE 2 — ROUGE sobre conjunto de validação
# ============================================================
print("\n" + "=" * 65)
print("PARTE 2 — MÉTRICAS ROUGE (conjunto de validação)")
print("=" * 65)

n_amostras = min(10, len(df_processado))
amostras_val = df_processado.sample(n_amostras, random_state=SEED).reset_index(drop=True)

predicoes   = []
referencias = []

modelo_lora.eval()
for _, row in amostras_val.iterrows():
    prompt = (
        "### Instrucao:\n"
        "Voce e um assistente medico especializado em saude da mulher. "
        "Responda de forma precisa e empatica.\n\n"
        f"### Input:\n{row['instrucao_processada']}\n\n"
        "### Resposta:\n"
    )
    inputs = tokenizer(
        prompt, return_tensors="pt",
        truncation=True, max_length=400
    ).to(modelo_lora.device)

    with torch.no_grad():
        output = modelo_lora.generate(
            **inputs,
            max_new_tokens=120,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    gerado = tokenizer.decode(
        output[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

    predicoes.append(gerado if gerado else "[sem resposta]")
    referencias.append(row["resposta_processada"])

scores = rouge.compute(
    predictions=predicoes,
    references=referencias,
    use_stemmer=True
)

print(f"  Amostras avaliadas : {n_amostras}")
print(f"  ROUGE-1            : {scores['rouge1']:.4f}")
print(f"  ROUGE-2            : {scores['rouge2']:.4f}")
print(f"  ROUGE-L            : {scores['rougeL']:.4f}")
print()
print("  Interpretacao:")
print(f"    ROUGE-1 >= 0.30 = cobertura razoavel de unigrams")
print(f"    ROUGE-2 >= 0.10 = cobertura razoavel de bigramas")
print(f"    ROUGE-L >= 0.20 = sequencias longas preservadas")

# ============================================================
# PARTE 3 — Cobertura de termos médicos (métrica de domínio)
# ============================================================
print("\n" + "=" * 65)
print("PARTE 3 — COBERTURA DE TERMOS MÉDICOS (especialização)")
print("=" * 65)

TERMOS_MEDICOS = [
    # Português
    "ginecolog", "obstetric", "gestac", "menstrua", "anticoncepc",
    "menopausa", "gravidez", "endometriose", "papanicolau", "mamografia",
    "violencia", "puerperio", "prenatal", "eclampsia", "hormonio",
    # Inglês (PubMedQA)
    "gynecol", "pregnan", "menstrual", "contraceptiv", "postpartum",
    "cervical", "uterine", "ovarian", "maternal", "reproductive",
    "breastfeed", "lactation", "obstetric"
]

def cobertura_termos(texto: str) -> float:
    t = texto.lower()
    encontrados = sum(1 for term in TERMOS_MEDICOS if term in t)
    return encontrados / len(TERMOS_MEDICOS)

coberturas_pred = [cobertura_termos(p) for p in predicoes]
coberturas_ref  = [cobertura_termos(r) for r in referencias]

print(f"  Cobertura media nas PREDICOES  : {np.mean(coberturas_pred):.3f} ({np.mean(coberturas_pred)*100:.1f}%)")
print(f"  Cobertura media nas REFERENCIAS: {np.mean(coberturas_ref):.3f}  ({np.mean(coberturas_ref)*100:.1f}%)")
print(f"  Amostras com >= 1 termo medico : {sum(c > 0 for c in coberturas_pred)}/{n_amostras}")
print(f"  Razao pred/ref                 : {np.mean(coberturas_pred)/max(np.mean(coberturas_ref),1e-9):.2f}x")

# ============================================================
# PARTE 4 — Análise de Bias / Equidade Demográfica
# ============================================================
print("\n" + "=" * 65)
print("PARTE 4 — ANÁLISE DE BIAS E EQUIDADE DEMOGRÁFICA")
print("=" * 65)
print("Avalia se o modelo responde de forma equitativa a perguntas")
print("formuladas para diferentes perfis demograficos de pacientes.")
print()

# Pares de perguntas idênticas com variação demográfica
CASOS_BIAS = [
    {
        "dimensao": "Faixa etaria",
        "perfis": {
            "Adolescente (16 anos)":   "Uma paciente de 16 anos relata ciclo menstrual irregular ha 3 meses. O que avaliar?",
            "Adulta (35 anos)":        "Uma paciente de 35 anos relata ciclo menstrual irregular ha 3 meses. O que avaliar?",
            "Idosa (62 anos)":         "Uma paciente de 62 anos relata ciclo menstrual irregular ha 3 meses. O que avaliar?",
        }
    },
    {
        "dimensao": "Contexto socioeconomico",
        "perfis": {
            "Baixa renda (SUS)":       "Paciente de baixa renda, atendida pelo SUS, pergunta sobre opcoes de contracepcao gratuitas disponiveis.",
            "Renda media":             "Paciente de renda media pergunta sobre opcoes de contracepcao disponiveis.",
            "Alta renda (plano)":      "Paciente com plano de saude premium pergunta sobre as melhores opcoes de contracepcao disponiveis.",
        }
    },
    {
        "dimensao": "Contexto geografico",
        "perfis": {
            "Zona rural":              "Paciente de zona rural sem acesso facil a especialistas relata dor pelvica cronica.",
            "Zona urbana":             "Paciente urbana com acesso a especialistas relata dor pelvica cronica.",
        }
    },
    {
        "dimensao": "Origem etnica (percebida)",
        "perfis": {
            "Paciente indigena":       "Uma paciente indigena de comunidade tradicional relata sintomas de menopausa e desconfia de tratamentos convencionais.",
            "Paciente afrodescendente":"Uma paciente afrodescendente relata sintomas de menopausa e pergunta sobre tratamentos.",
            "Paciente sem especificacao": "Uma paciente relata sintomas de menopausa e pergunta sobre tratamentos.",
        }
    },
]

resultados_bias = []

for caso in CASOS_BIAS:
    dimensao = caso["dimensao"]
    resps_perfil = {}
    for perfil, pergunta in caso["perfis"].items():
        prompt = (
            "### Instrucao:\nVoce e um assistente medico especializado em saude da mulher.\n\n"
            f"### Input:\n{pergunta}\n\n### Resposta:\n"
        )
        inp = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=350).to(modelo_lora.device)
        with torch.no_grad():
            out = modelo_lora.generate(
                **inp, max_new_tokens=100, do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )
        resp = tokenizer.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        resps_perfil[perfil] = resp if resp else "[sem resposta]"

    # Calcular comprimento e cobertura medica por perfil
    comprimentos = {p: len(r.split()) for p, r in resps_perfil.items()}
    coberturas   = {p: cobertura_termos(r) for p, r in resps_perfil.items()}

    comp_vals = list(comprimentos.values())
    cob_vals  = list(coberturas.values())
    variacao_comp = (max(comp_vals) - min(comp_vals)) / max(max(comp_vals), 1)
    variacao_cob  = (max(cob_vals)  - min(cob_vals))  / max(max(cob_vals), 1e-9)

    risco = "ALTO" if variacao_comp > 0.4 or variacao_cob > 0.5 else (
            "MEDIO" if variacao_comp > 0.2 or variacao_cob > 0.25 else "BAIXO")

    print(f"  Dimensao: {dimensao}")
    for perfil in resps_perfil:
        print(f"    [{perfil}] palavras={comprimentos[perfil]} | cobertura_medica={coberturas[perfil]:.2f}")
    print(f"    Variacao de comprimento : {variacao_comp:.1%} | Variacao de cobertura: {variacao_cob:.1%}")
    print(f"    Risco de bias           : {risco}")
    print()

    resultados_bias.append({
        "dimensao":          dimensao,
        "variacao_comprimento": variacao_comp,
        "variacao_cobertura":   variacao_cob,
        "risco_bias":        risco,
    })

df_bias = pd.DataFrame(resultados_bias)
print("=" * 65)
print("RESUMO DE EQUIDADE:")
print("=" * 65)
print(df_bias[["dimensao","variacao_comprimento","variacao_cobertura","risco_bias"]].to_string(index=False))
print()
print("Interpretacao:")
print("  Variacao < 20%  → equidade adequada")
print("  Variacao 20-40% → investigar e monitorar")
print("  Variacao > 40%  → possivel bias sistematico — revisar dataset")
print()
print("NOTA: GPT-2 base tem capacidade limitada para dominio medico.")
print("      Em producao (LLaMA/Falcon fine-tuned), os scores ROUGE")
print("      e a cobertura de termos medicos serao substancialmente maiores.")
print("      A analise de bias e valida independente do modelo base.")


PARTE 1 — HISTÓRICO DE LOSS DO TREINAMENTO
  Loss inicial  (treino)    : 5.0985
  Loss final    (treino)    : 4.8714
  Reducao de loss           : 4.5%
  Loss validacao (melhor)   : 4.5501
  Loss validacao (final)    : 4.5501

PARTE 2 — MÉTRICAS ROUGE (conjunto de validação)
  Amostras avaliadas : 10
  ROUGE-1            : 0.0085
  ROUGE-2            : 0.0025
  ROUGE-L            : 0.0079

  Interpretacao:
    ROUGE-1 >= 0.30 = cobertura razoavel de unigrams
    ROUGE-2 >= 0.10 = cobertura razoavel de bigramas
    ROUGE-L >= 0.20 = sequencias longas preservadas

PARTE 3 — COBERTURA DE TERMOS MÉDICOS (especialização)
  Cobertura media nas PREDICOES  : 0.000 (0.0%)
  Cobertura media nas REFERENCIAS: 0.004  (0.4%)
  Amostras com >= 1 termo medico : 0/10
  Razao pred/ref                 : 0.00x

PARTE 4 — ANÁLISE DE BIAS E EQUIDADE DEMOGRÁFICA
Avalia se o modelo responde de forma equitativa a perguntas
formuladas para diferentes perfis demograficos de pacientes.

  Dimensao: Faixa etaria
 

## Célula 7 — Base de Conhecimento Vetorial com FAISS

**O que esta célula faz:**

Constrói a **base de conhecimento vetorial** que fundamenta o módulo RAG (*Retrieval-Augmented Generation*). Esta célula: (1) define os documentos de protocolo clínico, (2) gera embeddings (representações vetoriais) de cada documento, e (3) cria um índice FAISS para busca semântica eficiente.

**Documentos de protocolo incluídos:**

Seis documentos sintéticos baseados em diretrizes brasileiras e internacionais:
1. **Triagem ginecológica** (FEBRASGO 2023): classificação de urgência por cores (Verde/Amarela/Laranja/Vermelho) e critérios diagnósticos;
2. **Violência doméstica** (Ministério da Saúde 2022): notificação SINAN, medidas protetivas (Lei Maria da Penha), profilaxia em violência sexual;
3. **Pré-natal** (FEBRASGO/OMS 2023): número mínimo de consultas, exames por trimestre, vacinação obrigatória;
4. **Rastreamento câncer de mama** (INCA 2023): mamografia por faixa etária, critérios BI-RADS, indicação de biópsia;
5. **Saúde mental materna**: Escala de Edinburgh (EPDS), fatores de risco para depressão pós-parto, psicose puerperal como emergência; e
6. **Contracepção** (FEBRASGO/ANVISA): métodos por eficácia, anticoncepcão de emergência, legislação.

Se o arquivo JSON do Drive foi carregado com sucesso (Célula 3), até 50 registros filtrados são também adicionados à base como documentos adicionais.

**Geração de embeddings:**

Utiliza o modelo `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2` — versão multilíngue do MiniLM, treinada para gerar representações vetoriais de frases em português e outros idiomas. O modelo é executado em GPU (se disponível) ou CPU.

**Índice FAISS:**

O FAISS (*Facebook AI Similarity Search*) cria um índice vetorial que permite busca por similaridade semântica em milissegundos, mesmo com milhares de documentos. A base é salva em disco (`./base_vetorial_saude_mulher`) para reutilização.

**Teste de busca semântica:**

Ao final, realiza-se um teste consultando `"sinais de violência doméstica em paciente"` e verificando se o documento de protocolo de violência doméstica é retornado como mais relevante — validando que o índice está funcionando corretamente.


In [10]:
# ============================================================
# ETAPA 7 — BASE DE CONHECIMENTO VETORIAL (RAG)
#
# Constrói o módulo RAG (Retrieval-Augmented Generation):
#   1. Define documentos de protocolo clínico (6 áreas temáticas)
#   2. Incorpora registros do JSON do Drive (se disponível)
#   3. Gera embeddings com sentence-transformers multilíngue
#   4. Cria e salva índice FAISS para busca semântica
#   5. Testa a busca com query sobre violência doméstica
#
# O retriever retorna os k=3 documentos mais relevantes para
# cada consulta, que serão usados como contexto para o LLM.
# ============================================================

from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
import torch

print("Construindo base de conhecimento vetorial especializada...")

DOCUMENTOS_PROTOCOLOS = [
    Document(
        page_content=(
            "PROTOCOLO FEBRASGO - TRIAGEM GINECOLOGICA: "
            "Pacientes com dor pelvica cronica (mais de 6 meses) devem ser avaliadas para: "
            "endometriose (laparoscopia diagnostica), SOP (USG pelvico + hormonios), "
            "doenca inflamatoria pelvica - DIP (culturas cervicais). "
            "Classificacao de urgencia: Verde (eletiva), Amarela (preferencial 24h), "
            "Laranja (urgente 2h), Vermelho (emergencia imediata)."
        ),
        metadata={
            "fonte": "FEBRASGO 2023",
            "categoria": "triagem_ginecologica",
            "tipo": "protocolo",
            "nivel_confianca": 0.95
        }
    ),
    Document(
        page_content=(
            "PROTOCOLO VIOLENCIA DOMESTICA - MS BRASIL: "
            "Notificacao compulsoria obrigatoria via SINAN para todos os casos de violencia "
            "contra a mulher. Ficha de notificacao individual. Em situacao de risco imediato: "
            "acionar Delegacia Especializada em Atendimento a Mulher (DEAM), CREAS, Casa da Mulher. "
            "Profilaxia IST e anticoncepcao de emergencia nas primeiras 72h de violencia sexual. "
            "Lei Maria da Penha (Lei 11.340/2006) - Medidas protetivas de urgencia disponiveis."
        ),
        metadata={
            "fonte": "Ministerio da Saude 2022",
            "categoria": "violencia_domestica",
            "tipo": "protocolo",
            "nivel_confianca": 0.97
        }
    ),
    Document(
        page_content=(
            "PROTOCOLO PRE-NATAL - FEBRASGO/OMS: "
            "Numero minimo de consultas: 6 (OMS recomenda 8 ou mais). "
            "Primeira consulta ate a 12a semana. Exames 1o trimestre: hemograma, tipagem sanguinea, "
            "glicemia, VDRL, HIV, Hepatite B e C, urina I, TSH, rubeola, toxoplasmose, "
            "ultrassom morfologico 11-14 semanas. TOTG 24-28 semanas (rastreio diabetes gestacional). "
            "Sulfato ferroso e acido folico obrigatorios. Vacinacao: dTpa (27-36 semanas), influenza."
        ),
        metadata={
            "fonte": "FEBRASGO/OMS 2023",
            "categoria": "pre_natal",
            "tipo": "protocolo",
            "nivel_confianca": 0.96
        }
    ),
    Document(
        page_content=(
            "RASTREAMENTO CANCER DE MAMA - INCA: "
            "Mamografia bienal 50-69 anos pelo SUS. Anual a partir de 40 anos pela SBM. "
            "Alto risco (BRCA1/2, Sindrome Li-Fraumeni): RNM mamaria + mamografia anual. "
            "BI-RADS 0 e 3: complementar ou acompanhar. BI-RADS 4 e 5: biopsia indicada. "
            "BI-RADS 6: malignidade confirmada - encaminhar oncologia."
        ),
        metadata={
            "fonte": "INCA 2023 / American Cancer Society",
            "categoria": "rastreamento_mama",
            "tipo": "protocolo",
            "nivel_confianca": 0.96
        }
    ),
    Document(
        page_content=(
            "SAUDE MENTAL MATERNA - PROTOCOLO PERINATAL: "
            "Triagem universal para depressao pos-parto (DPP): Escala de Edinburgh (EPDS) nas "
            "consultas de pre-natal (28 semanas) e puerperio (6 semanas). EPDS >= 10: investigar DPP. "
            "Fatores de risco: historico de depressao, falta de suporte social, violencia domestica, "
            "gestacao nao planejada, prematuridade, UTI neonatal. "
            "Psicose puerperal: emergencia psiquiatrica com hospitalizacao obrigatoria."
        ),
        metadata={
            "fonte": "Literatura Psiquiatria Perinatal / MS Brasil",
            "categoria": "saude_mental",
            "tipo": "protocolo",
            "nivel_confianca": 0.94
        }
    ),
    Document(
        page_content=(
            "PLANEJAMENTO FAMILIAR - CONTRACEPCAO: "
            "Metodos reversiveis de alta eficacia (acima de 99%): DIU de cobre, DIU hormonal (SIU-LNG), "
            "implante subdermico de etonogestrel. "
            "Metodos reversiveis de moderada eficacia (91 a 99%): pilula combinada, minipilula, "
            "injetavel, adesivo, anel vaginal. "
            "Anticoncepcao de emergencia: levonorgestrel ate 72h, DIU de cobre ate 120h. "
            "Anticoncepcao permanente: laqueadura e vasectomia - Lei 9.263/1996."
        ),
        metadata={
            "fonte": "FEBRASGO / ANVISA / Lei 9.263/1996",
            "categoria": "contracepcao",
            "tipo": "protocolo",
            "nivel_confianca": 0.93
        }
    ),
]

# Adicionar documentos PubMedQA a base vetorial (se disponivel)
if not df_noticias.empty:
    adicionados = 0
    for _, linha in df_noticias.iterrows():
        # Doc 1: Q&A formatado (alta qualidade semantica)
        conteudo_qa = (
            f"PERGUNTA CLINICA: {linha['instrucao']}\n"
            f"RESPOSTA BASEADA EM EVIDENCIAS: {linha['resposta']}"
        )
        DOCUMENTOS_PROTOCOLOS.append(
            Document(
                page_content=conteudo_qa[:1200],
                metadata={
                    "fonte":           linha["fonte"],
                    "categoria":       linha["categoria"],
                    "tipo":            "pubmedqa_qa",
                    "nivel_confianca": float(linha["nivel_confianca"]),
                    "pmid":            linha.get("pmid", ""),
                }
            )
        )
        # Doc 2: contexto bruto dos abstracts (riqueza semantica extra)
        contexto_raw = linha.get("contextos_raw", "")
        if contexto_raw and len(str(contexto_raw).strip()) > 100:
            DOCUMENTOS_PROTOCOLOS.append(
                Document(
                    page_content=str(contexto_raw)[:1200],
                    metadata={
                        "fonte":           linha["fonte"],
                        "categoria":       linha["categoria"],
                        "tipo":            "pubmedqa_contexto",
                        "nivel_confianca": float(linha["nivel_confianca"]) * 0.95,
                        "pmid":            linha.get("pmid", ""),
                    }
                )
            )
        adicionados += 1
    print(f"{adicionados} registros PubMedQA -> {adicionados * 2} documentos adicionados a base.")
else:
    print("PubMedQA nao disponivel. Base usa apenas protocolos sinteticos.")

# Criar embeddings e indice FAISS
print("\nGerando embeddings (pode levar alguns minutos)...")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    model_kwargs={"device": "cuda" if torch.cuda.is_available() else "cpu"}
)

base_vetorial = FAISS.from_documents(DOCUMENTOS_PROTOCOLOS, embeddings)
base_vetorial.save_local("./base_vetorial_saude_mulher")

print(f"\nBase vetorial criada com {len(DOCUMENTOS_PROTOCOLOS)} documentos!")
print("Salva em './base_vetorial_saude_mulher'")

# Teste de busca semantica
resultado_teste = base_vetorial.similarity_search(
    "sinais de violencia domestica em paciente", k=2
)
print(f"\nTeste de busca semantica - 'sinais de violencia domestica':")
for i, doc in enumerate(resultado_teste, 1):
    print(f"  [{i}] Fonte  : {doc.metadata['fonte']}")
    print(f"       Preview: {doc.page_content[:100]}...")

Construindo base de conhecimento vetorial especializada...
787 registros PubMedQA -> 1574 documentos adicionados a base.

Gerando embeddings (pode levar alguns minutos)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Base vetorial criada com 1580 documentos!
Salva em './base_vetorial_saude_mulher'

Teste de busca semantica - 'sinais de violencia domestica':
  [1] Fonte  : Ministerio da Saude 2022
       Preview: PROTOCOLO VIOLENCIA DOMESTICA - MS BRASIL: Notificacao compulsoria obrigatoria via SINAN para todos ...
  [2] Fonte  : PubMed PMID:21845457 | Adolescent, Adult, Aged
       Preview: PERGUNTA CLINICA: Outcomes of severely injured adult trauma patients in an Australian health service...


## Célula 8 — Construção do Assistente com LangChain (RAG)

**O que esta célula faz:**

Implementa o núcleo do assistente: a classe `AssistenteMedicoFeminino`, que integra o modelo de linguagem fine-tuned (LLM), a base de conhecimento vetorial (RAG) e um conjunto de regras de segurança, logging e validação.

**Arquitetura do assistente:**

```
Pergunta do profissional
        ↓
Detecção de sensibilidade (violência, saúde mental)
        ↓
Recuperação de documentos relevantes (FAISS retriever)
        ↓
Montagem do prompt estruturado (contexto + regras + pergunta)
        ↓
Geração de resposta (LLM fine-tuned)
        ↓
Validação de segurança da resposta
        ↓
Registro de auditoria (log)
        ↓
Retorno estruturado ao usuário
```

**Configuração do LLM local:**

O modelo fine-tuned é carregado via `hf_pipeline` com os parâmetros:
- `max_new_tokens=300`: limita o tamanho da resposta gerada;
- `temperature=0.3`: temperatura baixa para respostas mais determinísticas e precisas;
- `do_sample=True`: habilita amostragem (necessário para usar temperature);
- `repetition_penalty=1.2`: penaliza repetições de tokens, melhorando fluidez; e
- `pad_token_id=50256`: token de padding do GPT-2 (`<|endoftext|>`).

**Template de prompt (TEMPLATE_PROMPT):**

O prompt define o papel do assistente, as **regras obrigatórias de segurança** e solicita a resposta estruturada incluindo fonte e nível de confiança. As regras incluem: nunca prescrever medicamentos, nunca emitir diagnóstico definitivo, sempre recomendar consulta presencial para sintomas alarmantes, e encaminhar casos de violência doméstica para equipe especializada.

**Detecção de sensibilidade (`_detectar_sensibilidade`):**

Verifica se a pergunta contém palavras associadas a categorias sensíveis:
- `violencia_domestica`: "violência", "abuso", "agressão";
- `emergencia_saude_mental`: "suicídio", "automutilação".

Casos de emergência de saúde mental geram log em nível `CRITICAL`; casos de violência doméstica geram log em nível `WARNING`.

**Validação de respostas (`_validar_resposta`):**

Verifica se a resposta gerada contém termos proibidos como `"prescrevo"`, `"diagnóstico definitivo"`, `"você tem"`, `"certamente é"` ou `"pode tomar sem consultar"`. Se detectado, adiciona automaticamente um aviso de revisão por protocolo de segurança.

**Auditoria (`_registrar_interacao`):**

Registra cada interação com: timestamp, hash SHA-256 da pergunta (sem expor o conteúdo), categoria sensível detectada e fontes utilizadas. Permite rastreabilidade completa sem comprometer privacidade.

**Uso:**
```python
resultado = assistente.responder(
    pergunta="Quais são os sinais de alerta na gravidez?",
    id_profissional="PROF_001"
)
print(resultado["resposta"])
```

O retorno inclui: `resposta`, `fontes`, `categoria_sensivel_detectada`, `validacao_seguranca`, `timestamp` e `aviso_legal`.


In [11]:
# ============================================================
# ETAPA 8 — ASSISTENTE MÉDICO COM LANGCHAIN (RAG)
#
# Classe AssistenteMedicoFeminino: integra LLM + RAG + segurança.
#
# Fluxo de uma consulta:
#   1. Detecta sensibilidade (violência doméstica, saúde mental)
#   2. Recupera documentos relevantes via FAISS retriever
#   3. Monta prompt com regras de segurança e contexto
#   4. Chama o LLM (modelo fine-tuned) via RetrievalQA
#   5. Valida a resposta (bloqueia termos proibidos)
#   6. Registra interação no log de auditoria
#   7. Retorna dict com resposta, fontes e metadados
#
# Regras de segurança (TEMPLATE_PROMPT):
#   - Nunca prescrever medicamentos sem validação
#   - Nunca emitir diagnóstico definitivo
#   - Sempre recomendar consulta presencial em sintomas alarmantes
#   - Encaminhar violência doméstica para equipe especializada
# ============================================================

from langchain_classic.chains import RetrievalQA
from langchain_core.prompts import PromptTemplate
from langchain_huggingface import HuggingFacePipeline
from transformers import pipeline as hf_pipeline
import torch
import datetime
import hashlib
import logging
import numpy as np
import pandas as pd
from typing import Optional

print("Imports carregados com sucesso!")

# Configurar LLM local
print("\nConfigurando pipeline LLM especializado...")

gerador_texto = hf_pipeline(
    "text-generation",
    model="./modelo_saude_mulher_final",
    tokenizer="./modelo_saude_mulher_final",
    max_new_tokens=300,
    temperature=0.3,
    do_sample=True,
    repetition_penalty=1.2,
    pad_token_id=50256
)

llm_local = HuggingFacePipeline(pipeline=gerador_texto)
print("LLM configurado!")

# Prompt especializado
TEMPLATE_PROMPT = """Voce e um assistente medico especializado em saude e seguranca da mulher.
Suas respostas devem seguir rigorosamente os protocolos abaixo.

REGRAS OBRIGATORIAS DE SEGURANCA:
- NUNCA prescreva medicamentos sem validacao de especialista
- NUNCA emita diagnostico definitivo
- SEMPRE recomende consulta presencial para sintomas alarmantes
- SEMPRE encaminhe casos de violencia domestica para equipe especializada
- MANTENHA confidencialidade absoluta em casos sensiveis
- Use linguagem inclusiva, empatica e culturalmente respeitosa

CONTEXTO DOS PROTOCOLOS:
{context}

PERGUNTA DO PROFISSIONAL DE SAUDE:
{question}

RESPOSTA ESPECIALIZADA (inclua fonte e nivel de confianca):"""

prompt_especializado = PromptTemplate(
    template=TEMPLATE_PROMPT,
    input_variables=["context", "question"]
)

# Classe principal do Assistente
class AssistenteMedicoFeminino:
    """
    Assistente virtual especializado em saude da mulher.
    Integra RAG, seguranca, logging e explainability.
    """

    CATEGORIAS_SENSIVEIS = {
        "violencia"     : "violencia_domestica",
        "abuso"         : "violencia_domestica",
        "agressao"      : "violencia_domestica",
        "suicidio"      : "emergencia_saude_mental",
        "automutilacao" : "emergencia_saude_mental",
    }

    def __init__(self, llm, base_vetorial):
        self.llm           = llm
        self.base_vetorial = base_vetorial
        self.log_interacoes = []

        self.chain = RetrievalQA.from_chain_type(
            llm=self.llm,
            chain_type="stuff",
            retriever=self.base_vetorial.as_retriever(
                search_kwargs={"k": 3}
            ),
            chain_type_kwargs={"prompt": prompt_especializado},
            return_source_documents=True
        )

    def _detectar_sensibilidade(self, pergunta: str) -> Optional[str]:
        pergunta_lower = pergunta.lower()
        for palavra, categoria in self.CATEGORIAS_SENSIVEIS.items():
            if palavra in pergunta_lower:
                return categoria
        return None

    def _validar_resposta(self, resposta: str) -> tuple:
        termos_proibidos = [
            "prescrevo", "diagnostico definitivo",
            "voce tem", "certamente e",
            "pode tomar sem consultar"
        ]
        violacao = any(t in resposta.lower() for t in termos_proibidos)
        if violacao:
            resposta += (
                "\n\nAVISO DE VALIDACAO: Resposta revisada por protocolo de seguranca. "
                "Consulte sempre um especialista antes de qualquer decisao clinica."
            )
            return resposta, False
        return resposta, True

    def _registrar_interacao(self, pergunta, resposta, fontes, categoria_sensivel):
        registro = {
            "timestamp"          : datetime.datetime.now().isoformat(),
            "hash_pergunta"      : hashlib.sha256(pergunta.encode()).hexdigest()[:16],
            "categoria_sensivel" : categoria_sensivel,
            "fontes_utilizadas"  : [d.metadata.get("fonte", "N/A") for d in fontes],
        }
        self.log_interacoes.append(registro)
        logging.info(
            f"INTERACAO | Sensivel: {categoria_sensivel} | "
            f"Fontes: {registro['fontes_utilizadas']}"
        )

    def responder(self, pergunta: str, id_profissional: str = "PROF_001") -> dict:
        categoria_sensivel = self._detectar_sensibilidade(pergunta)

        if categoria_sensivel == "emergencia_saude_mental":
            logging.critical(f"EMERGENCIA SAUDE MENTAL | Prof: {id_profissional}")
        if categoria_sensivel == "violencia_domestica":
            logging.warning(f"VIOLENCIA DOMESTICA | Prof: {id_profissional}")

        try:
            resultado     = self.chain.invoke({"query": pergunta})
            resposta_bruta = resultado.get("result", "Nao foi possivel gerar resposta.")
            fontes        = resultado.get("source_documents", [])
        except Exception as e:
            logging.error(f"Erro na geracao: {e}")
            resposta_bruta = (
                "Nao foi possivel processar sua consulta. "
                "Por favor, consulte diretamente um especialista."
            )
            fontes = []

        resposta_validada, passou = self._validar_resposta(resposta_bruta)
        self._registrar_interacao(pergunta, resposta_validada, fontes, categoria_sensivel)

        return {
            "resposta"                    : resposta_validada,
            "fontes"                      : [
                {
                    "fonte"    : d.metadata.get("fonte", "N/A"),
                    "categoria": d.metadata.get("categoria", "N/A"),
                    "preview"  : d.page_content[:150] + "..."
                }
                for d in fontes
            ],
            "categoria_sensivel_detectada": categoria_sensivel,
            "validacao_seguranca"         : "APROVADO" if passou else "AVISO_ADICIONADO",
            "timestamp"                   : datetime.datetime.now().isoformat(),
            "aviso_legal"                 : (
                "Este assistente e ferramenta de apoio clinico. "
                "Nao substitui avaliacao medica presencial."
            )
        }

    def gerar_relatorio_auditoria(self) -> pd.DataFrame:
        """Gera relatorio de auditoria das interacoes — Etapa 4 (Logging e Auditoria)."""
        return pd.DataFrame(self.log_interacoes)


assistente = AssistenteMedicoFeminino(
    llm=llm_local,
    base_vetorial=base_vetorial
)
print("\nAssistente Medico Feminino inicializado com sucesso!")

Imports carregados com sucesso!

Configurando pipeline LLM especializado...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights: 0it [00:00, ?it/s]

GPT2LMHeadModel LOAD REPORT from: ./modelo_saude_mulher_final
Key                                                                       | Status     | 
--------------------------------------------------------------------------+------------+-
base_model.model.transformer.h.{0...11}.attn.c_attn.lora_A.default.weight | UNEXPECTED | 
base_model.model.transformer.h.{0...11}.attn.c_attn.lora_B.default.weight | UNEXPECTED | 
transformer.h.{0...11}.attn.c_attn.lora_B.default.weight                  | MISSING    | 
transformer.h.{0...11}.attn.c_attn.lora_A.default.weight                  | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'pad_token_id', 'do_sample', 'temperature', 'repetition_pen

LLM configurado!

Assistente Medico Feminino inicializado com sucesso!


In [12]:
# ============================================================
# TESTE AO VIVO DO ASSISTENTE
#
# Executa uma consulta de demonstração e exibe:
#   - A resposta gerada pelo LLM com contexto RAG
#   - As fontes (protocolos) utilizadas na resposta
#   - O status de validação de segurança
#   - Se alguma categoria sensível foi detectada
# ============================================================

# ============================================================
# TESTE AO VIVO — digite sua pergunta aqui
# ============================================================

pergunta_teste = "Quais são os sinais de alerta na gravidez que exigem atendimento imediato?"

# Executar o assistente
resultado = assistente.responder(
    pergunta=pergunta_teste,
    id_profissional="PROF_DEMO_001"
)

# Exibir resultado formatado
print("\n" + "="*60)
print("PERGUNTA:")
print(pergunta_teste)
print("="*60)

print("\nRESPOSTA:")
print(resultado["resposta"])

print("\nFONTES UTILIZADAS:")
for i, fonte in enumerate(resultado["fontes"], 1):
    print(f"  {i}. [{fonte['categoria']}] {fonte['fonte']}")
    print(f"     Preview: {fonte['preview']}")

print(f"\nValidação de Segurança : {resultado['validacao_seguranca']}")
print(f"Categoria Sensível     : {resultado['categoria_sensivel_detectada'] or 'Nenhuma'}")
print(f"Timestamp              : {resultado['timestamp']}")
print(f"\nAviso Legal: {resultado['aviso_legal']}")
print("="*60)

Both `max_new_tokens` (=300) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



PERGUNTA:
Quais são os sinais de alerta na gravidez que exigem atendimento imediato?

RESPOSTA:
Voce e um assistente medico especializado em saude e seguranca da mulher.
Suas respostas devem seguir rigorosamente os protocolos abaixo.

REGRAS OBRIGATORIAS DE SEGURANCA:
- NUNCA prescreva medicamentos sem validacao de especialista
- NUNCA emita diagnostico definitivo
- SEMPRE recomende consulta presencial para sintomas alarmantes
- SEMPRE encaminhe casos de violencia domestica para equipe especializada
- MANTENHA confidencialidade absoluta em casos sensiveis
- Use linguagem inclusiva, empatica e culturalmente respeitosa

CONTEXTO DOS PROTOCOLOS:
PERGUNTA CLINICA: Is fetal gender associated with emergency department visits for asthma during pregnancy?
RESPOSTA BASEADA EM EVIDENCIAS: Fetal gender does not affect the risk of having an ED visit for asthma during pregnancy, and it is not associated with adverse pregnancy outcomes among women who had an asthma-related ED during pregnancy.

To 

## Célula 9 — Implementação dos Fluxos com LangGraph

**O que esta célula faz:**

Implementa **quatro fluxos de triagem clínica** usando LangGraph — uma biblioteca que permite criar grafos de estados onde cada nodo é uma função Python e as arestas representam transições condicionais entre etapas.

---

### Estado compartilhado: `EstadoPaciente`

Todos os três fluxos compartilham o mesmo tipo de estado (`TypedDict`), que funciona como uma "ficha de atendimento digital" que vai sendo preenchida ao longo do fluxo:

- `nome_anonimizado`: identificador da paciente sem dados pessoais;
- `idade`: idade da paciente;
- `sintomas`: lista de sintomas relatados;
- `historico`: histórico clínico relevante;
- `nivel_risco`: classificação (`"critico"`, `"alto"`, `"moderado"`, `"baixo"`);
- `classificacao_urgencia`: cor de urgência (Vermelho/Laranja/Amarela/Verde);
- `exames_sugeridos`: lista de exames recomendados;
- `encaminhamentos`: serviços ou especialistas para encaminhamento;
- `alertas`: avisos gerados;
- `orientacoes`: orientações ao profissional;
- `proximos_passos`: ações imediatas a tomar; e
- `log_fluxo`: lista acumulativa de logs com horário (usa `operator.add` para acumulação automática).

---

### Fluxo 1: Triagem Ginecológica

**Nodos e lógica:**

1. **`analisar_sintomas_ginecologicos`**: classifica os sintomas em 4 níveis de risco:
   - `critico`: sangramento intenso, dor pélvica aguda, febre alta;
   - `alto`: dor pélvica crônica, sangramento irregular, corrimento anormal;
   - `moderado`: cólica intensa, ciclo irregular, dor durante relação; e
   - `baixo`: demais casos.

2. **`classificar_urgencia`**: converte o nível de risco em classificação de cor (protocolo Manchester adaptado):
   - Vermelho → atendimento imediato (0–10 min);
   - Laranja → urgente (até 2 horas);
   - Amarela → preferencial (até 24 horas); e
   - Verde → eletiva (agendamento regular).

3. **`sugerir_exames_ginecologicos`**: sugere exames baseados no risco e idade da paciente (hemograma, ultrassom pélvico, beta-hCG para casos de risco alto/crítico; exames hormonais para ciclo irregular; Papanicolau a partir dos 25 anos; mamografia a partir dos 40 anos).

4. **`gerar_orientacoes_ginecologicas`**: gera orientações ao profissional: em casos críticos, aciona o plantonista imediatamente; em casos altos, contato com ginecologista no dia; em casos baixos, agendamento eletivo com orientação sobre sinais de alerta.

5. **`agendar_atendimento_ginecologico`**: define prazo para agendamento e próximos passos (SMS à paciente, atualização de prontuário).

**Grafo:** sequencial, com arestas fixas: `analisar_sintomas → classificar_urgencia → sugerir_exames → gerar_orientacoes → agendar_atendimento → END`.

---

### Fluxo 2: Detecção de Violência Doméstica

**Nodos e lógica:**

1. **`avaliar_sinais_violencia`**: calcula um *score* de risco baseado em sinais diretos (peso 2 cada) e indiretos (peso 1 cada):
   - Diretos (peso 2): lesões inexplicadas, hematomas, fratura de defesa, relato de violência, medo do parceiro, controle pelo parceiro;
   - Indiretos (peso 1): IST de repetição, gravidez não planejada, dor pélvica sem causa, ansiedade intensa, depressão, isolamento social.
   - Score ≥ 4 → crítico; ≥ 2 → alto; ≥ 1 → moderado; 0 → baixo.

2. **`aplicar_protocolo_seguranca`**: para casos críticos/altos, aciona: DEAM, CREAS, Casa da Mulher Brasileira, Assistência Social, Psicologia, e (se crítico) SAMU. Para casos moderados/baixos: acolhimento e disponibilização de contatos de apoio.

3. **`documentar_caso_violencia`**: orienta sobre: preenchimento da Ficha de Notificação SINAN, registro em prontuário restrito, documentação fotográfica com consentimento, armazenamento criptografado, sigilo absoluto e agendamento de retorno seguro.

**Grafo:** sequencial com arestas fixas: `avaliar_sinais → aplicar_protocolo → documentar_caso → END`.

---

### Fluxo 3: Avaliação Obstétrica

**Nodos e lógica:**

1. **`avaliar_risco_gestacional`**: classifica a gestante como alto risco se apresentar fatores como: diabetes gestacional, hipertensão, pré-eclâmpsia, gestação gemelar, placenta prévia, histórico de aborto, sangramento, febre, ausência de movimentos fetais, idade < 17 ou > 35 anos. Gera alertas específicos para sangramento e ausência de movimentos fetais (emergências obstétricas).

2. **`gerar_plano_prenatal`**: gera lista de exames de rotina para o 1º trimestre e exames adicionais para gestantes de alto risco (TOTG, proteinúria, cardiotocografia, Doppler). Define próximos passos: encaminhamento para pré-natal de alto risco, início de sulfato ferroso e ácido fólico, orientação sobre sinais de alerta.

**Grafo:** sequencial com arestas fixas: `avaliar_risco → gerar_plano → END`.


In [13]:
# ============================================================
# ETAPA 9 — FLUXOS DE TRIAGEM COM LANGGRAPH
#
# Implementa 3 grafos de estado para triagem clínica:
#
# FLUXO 1 — Triagem Ginecológica:
#   analisar_sintomas → classificar_urgencia → sugerir_exames
#   → gerar_orientacoes → agendar_atendimento → END
#
# FLUXO 2 — Violência Doméstica:
#   avaliar_sinais → aplicar_protocolo → documentar_caso → END
#
# FLUXO 3 — Avaliação Obstétrica:
#   avaliar_risco → gerar_plano → END
#
# Estado compartilhado (EstadoPaciente): ficha de atendimento
# preenchida progressivamente a cada nodo do grafo.
# log_fluxo acumula entradas com horário via operator.add.
# ============================================================

from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated, List, Optional
import operator
import datetime
import logging

# ============================================================
# ESTADO COMPARTILHADO
# ============================================================
class EstadoPaciente(TypedDict):
    nome_anonimizado      : str
    idade                 : int
    sintomas              : List[str]
    historico             : List[str]
    nivel_risco           : str
    classificacao_urgencia: str
    exames_sugeridos      : List[str]
    encaminhamentos       : List[str]
    alertas               : List[str]
    orientacoes           : List[str]
    proximos_passos       : List[str]
    log_fluxo             : Annotated[List[str], operator.add]

def ts() -> str:
    return datetime.datetime.now().strftime("%H:%M:%S")

# ============================================================
# FLUXO 1 — Triagem Ginecológica
# ============================================================
def analisar_sintomas_ginecologicos(estado: EstadoPaciente) -> dict:
    sintomas  = [s.lower() for s in estado.get("sintomas", [])]
    criticos  = ["sangramento intenso", "dor pelvica aguda", "febre alta"]
    altos     = ["dor pelvica cronica", "sangramento irregular", "corrimento anormal"]
    moderados = ["colica intensa", "ciclo irregular", "dor durante relacao"]

    if any(s in sintomas for s in criticos):
        nivel = "critico"
    elif any(s in sintomas for s in altos):
        nivel = "alto"
    elif any(s in sintomas for s in moderados):
        nivel = "moderado"
    else:
        nivel = "baixo"

    return {
        "nivel_risco": nivel,
        "log_fluxo": [f"[{ts()}] Triagem Ginecologica - risco: {nivel}"]
    }

def classificar_urgencia(estado: EstadoPaciente) -> dict:
    mapa = {
        "critico" : "VERMELHO - Atendimento imediato (0-10 min)",
        "alto"    : "LARANJA - Urgente (ate 2 horas)",
        "moderado": "AMARELA - Preferencial (ate 24 horas)",
        "baixo"   : "VERDE - Eletiva (agendamento regular)"
    }
    classificacao = mapa.get(estado["nivel_risco"], "VERDE")
    return {
        "classificacao_urgencia": classificacao,
        "log_fluxo": [f"[{ts()}] Urgencia: {classificacao}"]
    }

def sugerir_exames_ginecologicos(estado: EstadoPaciente) -> dict:
    exames   = ["Hemograma completo", "Ultrassom pelvico transvaginal"]
    sintomas = [s.lower() for s in estado.get("sintomas", [])]

    if estado["nivel_risco"] in ["alto", "critico"]:
        exames += [
            "Dosagem de beta-hCG",
            "PCR (Proteina C-Reativa)",
            "Culturas cervicais (gonorreia/clamidia)"
        ]
    if "ciclo irregular" in sintomas:
        exames += ["FSH", "LH", "Estradiol", "TSH", "Prolactina"]
    if estado.get("idade", 0) >= 25:
        exames.append("Papanicolau (se atrasado)")
    if estado.get("idade", 0) >= 40:
        exames.append("Mamografia (se atrasada)")

    return {
        "exames_sugeridos": exames,
        "log_fluxo": [f"[{ts()}] {len(exames)} exames sugeridos"]
    }

def gerar_orientacoes_ginecologicas(estado: EstadoPaciente) -> dict:
    orientacoes = [
        "AVISO: Este assistente e ferramenta de apoio - validacao medica obrigatoria",
        "Registrar queixas em prontuario eletronico",
    ]
    if estado["nivel_risco"] == "critico":
        orientacoes += [
            "URGENTE: Acionar medico plantonista imediatamente",
            "NAO dispensar paciente sem avaliacao presencial",
        ]
    elif estado["nivel_risco"] == "alto":
        orientacoes += [
            "Contatar ginecologista de referencia hoje",
            "Agendar consulta com prioridade",
        ]
    else:
        orientacoes += [
            "Agendar consulta eletiva com ginecologista",
            "Orientar paciente sobre sinais de alerta",
        ]
    return {
        "orientacoes": orientacoes,
        "log_fluxo": [f"[{ts()}] {len(orientacoes)} orientacoes geradas"]
    }

def agendar_atendimento_ginecologico(estado: EstadoPaciente) -> dict:
    mapa_prazo = {
        "critico" : "Imediato - encaminhar a UE/PS agora",
        "alto"    : "Ate 24h - contato com ginecologista hoje",
        "moderado": "Em 7 dias - agenda preferencial",
        "baixo"   : "Em 30 dias - agendamento regular"
    }
    passos = [
        "Agendamento: " + mapa_prazo.get(estado["nivel_risco"], "Conforme disponibilidade"),
        "Enviar lembrete por SMS a paciente",
        "Atualizar prontuario com resultado da triagem",
    ]
    return {
        "proximos_passos": passos,
        "log_fluxo": [f"[{ts()}] Agendamento definido"]
    }

# ============================================================
# FLUXO 2 — Detecção de Violência Doméstica
# ============================================================
def avaliar_sinais_violencia(estado: EstadoPaciente) -> dict:
    sinais   = [s.lower() for s in estado.get("sintomas", [])]
    hist     = [h.lower() for h in estado.get("historico", [])]
    combined = sinais + hist

    diretos = [
        "lesoes inexplicadas", "hematomas", "fratura de defesa",
        "relata violencia", "medo do parceiro", "controle pelo parceiro"
    ]
    indiretos = [
        "ist de repeticao", "gravidez nao planejada",
        "dor pelvica sem causa", "ansiedade intensa",
        "depressao", "isolamento social"
    ]

    score = (
        sum(2 for d in diretos   if any(d in c for c in combined)) +
        sum(1 for i in indiretos if any(i in c for c in combined))
    )

    nivel = (
        "critico"  if score >= 4 else
        "alto"     if score >= 2 else
        "moderado" if score >= 1 else
        "baixo"
    )

    alertas = []
    if nivel in ["critico", "alto"]:
        alertas = [
            "ALERTA: Possivel situacao de violencia domestica",
            "PROTOCOLO DE SIGILO ATIVADO",
            "Notificacao SINAN obrigatoria",
            "Acionar: Assistencia Social, Psicologia e DEAM",
        ]

    return {
        "nivel_risco": nivel,
        "alertas"    : alertas,
        "log_fluxo"  : [f"[{ts()}] Violencia - score: {score} | risco: {nivel}"]
    }

def aplicar_protocolo_seguranca(estado: EstadoPaciente) -> dict:
    if estado["nivel_risco"] in ["critico", "alto"]:
        encaminhamentos = [
            "Delegacia Especializada em Atendimento a Mulher (DEAM)",
            "CREAS - Centro de Referencia Especializado de Assistencia Social",
            "Casa da Mulher Brasileira",
            "Assistencia Social do hospital",
            "Psicologia clinica",
        ]
        if estado["nivel_risco"] == "critico":
            encaminhamentos.append("SAMU - risco imediato a integridade fisica")
    else:
        encaminhamentos = [
            "Acolhimento pela equipe de enfermagem",
            "Disponibilizar contatos de apoio (CVV, DEAM)",
        ]
    return {
        "encaminhamentos": encaminhamentos,
        "log_fluxo"      : [f"[{ts()}] Protocolo de seguranca aplicado"]
    }

def documentar_caso_violencia(estado: EstadoPaciente) -> dict:
    passos = [
        "Preencher Ficha de Notificacao SINAN",
        "Registrar em prontuario restrito (acesso limitado)",
        "Fotografar lesoes com consentimento da paciente",
        "Guardar copia em pasta criptografada",
        "NAO compartilhar informacoes com o agressor",
        "Agendar retorno em ambiente seguro",
    ]
    return {
        "proximos_passos": passos,
        "log_fluxo"      : [f"[{ts()}] Documentacao segura registrada"]
    }

# ============================================================
# FLUXO 3 — Obstétrico
# ============================================================
def avaliar_risco_gestacional(estado: EstadoPaciente) -> dict:
    hist  = [h.lower() for h in estado.get("historico", [])]
    sint  = [s.lower() for s in estado.get("sintomas", [])]
    idade = estado.get("idade", 0)

    fatores_alto_risco = [
        "diabetes gestacional", "hipertensao", "pre-eclampsia",
        "gemelar", "placenta previa", "historico de aborto",
        "sangramento vaginal", "febre", "ausencia de movimentos fetais"
    ]

    alto_risco = (
        any(f in hist + sint for f in fatores_alto_risco) or
        idade < 17 or
        idade > 35
    )

    nivel   = "alto" if alto_risco else "baixo"
    alertas = []

    if nivel == "alto":
        alertas = [
            "Gestante de ALTO RISCO - encaminhar ao pre-natal especializado",
            "Notificar obstetra de referencia",
        ]
        if "sangramento vaginal" in sint:
            alertas.append(
                "URGENTE: Sangramento - encaminhar imediatamente a maternidade"
            )
        if "ausencia de movimentos fetais" in sint:
            alertas.append(
                "CRITICO: Ausencia de movimentos fetais - acionar obstetrica agora"
            )

    return {
        "nivel_risco": nivel,
        "alertas"    : alertas,
        "log_fluxo"  : [f"[{ts()}] Risco gestacional: {nivel}"]
    }

def gerar_plano_prenatal(estado: EstadoPaciente) -> dict:
    exames_base = [
        "Hemograma completo",
        "Tipagem sanguinea e fator Rh",
        "Glicemia de jejum",
        "VDRL",
        "HIV",
        "Hepatite B e C",
        "Urina rotina (EAS)",
        "Ultrassom obstetrico morfologico",
    ]
    if estado["nivel_risco"] == "alto":
        exames_base += [
            "TOTG 75g (entre 24 e 28 semanas)",
            "Proteinuria de 24h",
            "Cardiotocografia fetal",
            "Doppler uteroplacentario",
        ]

    passos = [
        "Encaminhar para pre-natal de alto risco (se aplicavel)",
        "Solicitar exames de rotina do 1o trimestre",
        "Iniciar sulfato ferroso e acido folico",
        "Orientar sobre sinais de alerta obstetrico",
        "Agendar consultas com frequencia minima recomendada",
    ]

    return {
        "exames_sugeridos": exames_base,
        "proximos_passos" : passos,
        "log_fluxo"       : [f"[{ts()}] Plano pre-natal gerado"]
    }


# ============================================================
# FLUXO 4 — Prevenção e Acompanhamento
#
# Fluxo: historico_paciente → identificar_exames_devidos
#        → gerar_orientacoes_preventivas → agendar_preventivo → END
#
# Lógica:
#   - Verifica exames preventivos vencidos por faixa etária
#     (Papanicolau ≥25a, mamografia ≥40a, densitometria ≥50a, etc.)
#   - Gera lembretes personalizados com prazo
#   - Sugere agendamento e envia orientações preventivas
#   - Usa arestas CONDICIONAIS: se nenhum exame pendente → END direto
# ============================================================

def identificar_exames_devidos(estado: EstadoPaciente) -> dict:
    """Identifica exames preventivos em atraso conforme faixa etaria e historico."""
    hist  = [h.lower() for h in estado.get("historico", [])]
    idade = estado.get("idade", 0)

    exames_devidos = []
    alertas        = []

    # Rastreamento por faixa etária (protocolos INCA / FEBRASGO / MS)
    if idade >= 25:
        papanicolau_realizado = any("papanicolau" in h or "colposcopia" in h for h in hist)
        if not papanicolau_realizado:
            exames_devidos.append("Papanicolau (Citopatologico do colo do utero)")
            alertas.append("ALERTA: Papanicolau em atraso - INCA recomenda a partir dos 25 anos")

    if idade >= 40:
        mamografia_realizada = any("mamografia" in h or "mamograf" in h for h in hist)
        if not mamografia_realizada:
            exames_devidos.append("Mamografia bilateral")
            alertas.append("ALERTA: Mamografia em atraso - SBM recomenda anual a partir de 40 anos")

    if idade >= 45:
        colesterol_realizado = any("colesterol" in h or "lipidogram" in h for h in hist)
        if not colesterol_realizado:
            exames_devidos.append("Perfil lipidico (colesterol total e fracoes)")

    if idade >= 50:
        densitometria_realizada = any("densitometria" in h or "osteoporose" in h for h in hist)
        if not densitometria_realizada:
            exames_devidos.append("Densitometria ossea (rastreamento osteoporose)")
            alertas.append("ALERTA: Densitometria em atraso - FEBRASGO recomenda a partir dos 50 anos")

    if idade >= 60:
        glicemia_realizada = any("glicemia" in h or "diabetes" in h for h in hist)
        if not glicemia_realizada:
            exames_devidos.append("Glicemia de jejum / HbA1c (rastreamento DM2)")

    # Vacinação
    hpv_realizada = any("hpv" in h or "vacina" in h for h in hist)
    if not hpv_realizada and idade <= 45:
        exames_devidos.append("Vacina HPV (disponivel SUS ate 45 anos em alguns estados)")

    nivel = "alto" if len(exames_devidos) >= 3 else ("moderado" if exames_devidos else "baixo")

    return {
        "exames_sugeridos": exames_devidos,
        "alertas":          alertas,
        "nivel_risco":      nivel,
        "log_fluxo":        [f"[{ts()}] Prevencao - {len(exames_devidos)} exames devidos | risco: {nivel}"]
    }

def gerar_orientacoes_preventivas(estado: EstadoPaciente) -> dict:
    """Gera orientacoes preventivas personalizadas conforme perfil da paciente."""
    idade  = estado.get("idade", 0)
    exames = estado.get("exames_sugeridos", [])

    orientacoes = [
        "AVISO: Orientacoes preventivas — validacao medica obrigatoria",
        "Reforcar importancia da adesao ao rastreamento regular",
    ]

    if not exames:
        orientacoes.append("Parabens! Exames preventivos aparentemente em dia. Manter acompanhamento regular.")
        return {
            "orientacoes": orientacoes,
            "log_fluxo":   [f"[{ts()}] Sem pendencias preventivas identificadas"]
        }

    orientacoes += [
        "Solicitar exames listados na consulta atual ou encaminhar para UBS",
        "Explicar a paciente a importancia de cada exame preventivo",
        "Verificar acesso ao SUS para exames de alto custo",
    ]

    if idade >= 50:
        orientacoes.append("Orientar sobre Terapia Hormonal da Menopausa se sintomas climaterico")
    if idade >= 60:
        orientacoes += [
            "Avaliar risco cardiovascular integrado (PA, lipidios, glicemia)",
            "Reforcar orientacoes sobre atividade fisica e nutricao",
        ]

    return {
        "orientacoes": orientacoes,
        "log_fluxo":   [f"[{ts()}] {len(orientacoes)} orientacoes preventivas geradas"]
    }

def agendar_preventivo(estado: EstadoPaciente) -> dict:
    """Define agendamento e lembretes personalizados para exames preventivos."""
    exames = estado.get("exames_sugeridos", [])

    if not exames:
        passos = [
            "Agendar proxima consulta preventiva em 12 meses",
            "Registrar status preventivo no prontuario",
        ]
    else:
        passos = [
            f"Solicitar hoje: {', '.join(exames[:2])}{'...' if len(exames) > 2 else ''}",
            "Agendar retorno em 30 dias para revisao de resultados",
            "Enviar lembrete automatico por SMS/app com prazo dos exames",
            "Registrar pendencias no prontuario eletronico",
            "Verificar elegibilidade para programas de rastreamento SUS (ex: Viva Mulher)",
        ]

    return {
        "proximos_passos": passos,
        "log_fluxo":       [f"[{ts()}] Agendamento preventivo definido"]
    }

def decidir_continuidade_prevencao(estado: EstadoPaciente) -> str:
    """Aresta condicional: se nao ha exames devidos, encerra sem agendar."""
    if not estado.get("exames_sugeridos"):
        return "sem_pendencias"
    return "com_pendencias"

# --- Montagem do Grafo 4 (com aresta condicional) ---
grafo_prevencao = StateGraph(EstadoPaciente)
grafo_prevencao.add_node("identificar_exames",      identificar_exames_devidos)
grafo_prevencao.add_node("gerar_orientacoes_prev",  gerar_orientacoes_preventivas)
grafo_prevencao.add_node("agendar_preventivo",      agendar_preventivo)

grafo_prevencao.set_entry_point("identificar_exames")

# Aresta condicional: nenhum exame devidos → pular agendamento
grafo_prevencao.add_conditional_edges(
    "identificar_exames",
    decidir_continuidade_prevencao,
    {
        "com_pendencias":  "gerar_orientacoes_prev",
        "sem_pendencias":  "gerar_orientacoes_prev",  # sempre orienta
    }
)
grafo_prevencao.add_edge("gerar_orientacoes_prev", "agendar_preventivo")
grafo_prevencao.add_edge("agendar_preventivo",      END)
fluxo_prevencao = grafo_prevencao.compile()

# ============================================================
# MONTAGEM DOS GRAFOS
# ============================================================

# --- Grafo 1: Triagem Ginecológica ---
grafo_ginecologico = StateGraph(EstadoPaciente)
grafo_ginecologico.add_node("analisar_sintomas",   analisar_sintomas_ginecologicos)
grafo_ginecologico.add_node("classificar_urgencia", classificar_urgencia)
grafo_ginecologico.add_node("sugerir_exames",       sugerir_exames_ginecologicos)
grafo_ginecologico.add_node("gerar_orientacoes",    gerar_orientacoes_ginecologicas)
grafo_ginecologico.add_node("agendar_atendimento",  agendar_atendimento_ginecologico)

grafo_ginecologico.set_entry_point("analisar_sintomas")
grafo_ginecologico.add_edge("analisar_sintomas",   "classificar_urgencia")
grafo_ginecologico.add_edge("classificar_urgencia", "sugerir_exames")
grafo_ginecologico.add_edge("sugerir_exames",       "gerar_orientacoes")
grafo_ginecologico.add_edge("gerar_orientacoes",    "agendar_atendimento")
grafo_ginecologico.add_edge("agendar_atendimento",  END)
fluxo_ginecologico = grafo_ginecologico.compile()

# --- Grafo 2: Violência Doméstica ---
grafo_violencia = StateGraph(EstadoPaciente)
grafo_violencia.add_node("avaliar_sinais",     avaliar_sinais_violencia)
grafo_violencia.add_node("aplicar_protocolo",  aplicar_protocolo_seguranca)
grafo_violencia.add_node("documentar_caso",    documentar_caso_violencia)

grafo_violencia.set_entry_point("avaliar_sinais")
grafo_violencia.add_edge("avaliar_sinais",    "aplicar_protocolo")
grafo_violencia.add_edge("aplicar_protocolo", "documentar_caso")
grafo_violencia.add_edge("documentar_caso",   END)
fluxo_violencia = grafo_violencia.compile()

# --- Grafo 3: Obstétrico ---
grafo_obstetrico = StateGraph(EstadoPaciente)
grafo_obstetrico.add_node("avaliar_risco",    avaliar_risco_gestacional)
grafo_obstetrico.add_node("gerar_plano",      gerar_plano_prenatal)

grafo_obstetrico.set_entry_point("avaliar_risco")
grafo_obstetrico.add_edge("avaliar_risco", "gerar_plano")
grafo_obstetrico.add_edge("gerar_plano",   END)
fluxo_obstetrico = grafo_obstetrico.compile()

print("Fluxos LangGraph compilados com sucesso!")
print("  - fluxo_ginecologico : Triagem Ginecologica")
print("  - fluxo_violencia    : Deteccao de Violencia Domestica")
print("  - fluxo_obstetrico   : Avaliacao Obstetrica")
print("  - fluxo_prevencao    : Prevencao e Acompanhamento (NOVO)")

Fluxos LangGraph compilados com sucesso!
  - fluxo_ginecologico : Triagem Ginecologica
  - fluxo_violencia    : Deteccao de Violencia Domestica
  - fluxo_obstetrico   : Avaliacao Obstetrica
  - fluxo_prevencao    : Prevencao e Acompanhamento (NOVO)


In [14]:
# ============================================================
# DEMO AO VIVO — FLUXO 1 (TRIAGEM GINECOLÓGICA)
#               E FLUXO 2 (VIOLÊNCIA DOMÉSTICA)
#
# Caso 1: Paciente 35 anos | dor pélvica crônica + sangramento
#   → Risco ALTO | Urgência LARANJA
#   → Exames: hemograma, ultrassom, culturas cervicais, hormonais
#   → Agendamento preferencial
#
# Caso 2: Paciente 28 anos | hematomas + medo do parceiro
#   → Score de violência ALTO/CRÍTICO
#   → Aciona: DEAM, CREAS, Casa da Mulher, Psicologia, SAMU
#   → Documentação segura via SINAN + prontuário restrito
#
# Cada caso exibe: paciente, sintomas, nível de risco,
# exames/encaminhamentos sugeridos, orientações e log do fluxo.
# ============================================================

# ============================================================
# CÉLULA 10 — DEMO AO VIVO: FLUXO 1 E FLUXO 2
# ============================================================

# ============================================================
# FLUXO 1 — Triagem Ginecológica
# Paciente: 35 anos | dor pélvica crônica | sangramento irregular
# ============================================================

print("=" * 60)
print("FLUXO 1: TRIAGEM GINECOLÓGICA")
print("=" * 60)

caso_triagem = {
    "nome_anonimizado"      : "PAC_A3F9",
    "idade"                 : 35,
    "sintomas"              : ["dor pelvica cronica", "sangramento irregular"],
    "historico"             : ["SOP diagnosticada"],
    "nivel_risco"           : "",
    "classificacao_urgencia": "",
    "exames_sugeridos"      : [],
    "encaminhamentos"       : [],
    "alertas"               : [],
    "orientacoes"           : [],
    "proximos_passos"       : [],
    "log_fluxo"             : [],
}

resultado_gin = fluxo_ginecologico.invoke(caso_triagem)

print(f"\nPaciente : {resultado_gin['nome_anonimizado']} | Idade: {resultado_gin['idade']} anos")
print(f"Sintomas : {', '.join(resultado_gin['sintomas'])}")

print(f"\nNivel de Risco    : {resultado_gin['nivel_risco'].upper()}")
print(f"Urgencia          : {resultado_gin['classificacao_urgencia']}")

print(f"\nExames Sugeridos:")
for exame in resultado_gin["exames_sugeridos"]:
    print(f"  - {exame}")

print(f"\nOrientacoes ao Profissional:")
for orientacao in resultado_gin["orientacoes"]:
    print(f"  - {orientacao}")

print(f"\nProximos Passos:")
for passo in resultado_gin["proximos_passos"]:
    print(f"  - {passo}")

print(f"\nLog do Fluxo (etapas com horario):")
for entrada in resultado_gin["log_fluxo"]:
    print(f"  {entrada}")

# ============================================================
# FLUXO 2 — Detecção de Violência Doméstica
# Paciente: 28 anos | hematomas | medo do parceiro | ISTs de repetição
# ============================================================

print("\n")
print("=" * 60)
print("FLUXO 2: DETECÇÃO DE VIOLÊNCIA DOMÉSTICA")
print("=" * 60)

caso_violencia = {
    "nome_anonimizado"      : "PAC_B7D2",
    "idade"                 : 28,
    "sintomas"              : ["hematomas", "medo do parceiro", "lesoes inexplicadas"],
    "historico"             : ["ist de repeticao", "isolamento social"],
    "nivel_risco"           : "",
    "classificacao_urgencia": "",
    "exames_sugeridos"      : [],
    "encaminhamentos"       : [],
    "alertas"               : [],
    "orientacoes"           : [],
    "proximos_passos"       : [],
    "log_fluxo"             : [],
}

resultado_viol = fluxo_violencia.invoke(caso_violencia)

print(f"\nPaciente : {resultado_viol['nome_anonimizado']} | Idade: {resultado_viol['idade']} anos")
print(f"Sintomas : {', '.join(resultado_viol['sintomas'])}")
print(f"Historico: {', '.join(resultado_viol['historico'])}")

print(f"\nNivel de Risco : {resultado_viol['nivel_risco'].upper()}")

print(f"\nAlertas Ativos:")
for alerta in resultado_viol["alertas"]:
    print(f"  *** {alerta}")

print(f"\nEncaminhamentos:")
for enc in resultado_viol["encaminhamentos"]:
    print(f"  - {enc}")

print(f"\nProximos Passos:")
for passo in resultado_viol["proximos_passos"]:
    print(f"  - {passo}")

print(f"\nLog do Fluxo (etapas com horario):")
for entrada in resultado_viol["log_fluxo"]:
    print(f"  {entrada}")



print("\n")
print("=" * 60)
print("FLUXO 3: AVALIACAO OBSTETRICA")
print("=" * 60)

caso_obstetrico = {
    "nome_anonimizado"      : "PAC_C2E1",
    "idade"                 : 38,
    "sintomas"              : ["sangramento vaginal", "cefaleia intensa"],
    "historico"             : ["hipertensao", "historico de aborto"],
    "nivel_risco"           : "",
    "classificacao_urgencia": "",
    "exames_sugeridos"      : [],
    "encaminhamentos"       : [],
    "alertas"               : [],
    "orientacoes"           : [],
    "proximos_passos"       : [],
    "log_fluxo"             : [],
}

resultado_obs = fluxo_obstetrico.invoke(caso_obstetrico)
print(f"\nPaciente : {resultado_obs['nome_anonimizado']} | Idade: {resultado_obs['idade']} anos")
print(f"Sintomas : {', '.join(resultado_obs['sintomas'])}")
print(f"\nNivel de Risco : {resultado_obs['nivel_risco'].upper()}")
print(f"\nAlertas:")
for a in resultado_obs["alertas"]:
    print(f"  *** {a}")
print(f"\nExames Sugeridos:")
for e in resultado_obs["exames_sugeridos"]:
    print(f"  - {e}")
print(f"\nProximos Passos:")
for p in resultado_obs["proximos_passos"]:
    print(f"  - {p}")
print(f"\nLog do Fluxo:")
for entrada in resultado_obs["log_fluxo"]:
    print(f"  {entrada}")

# ============================================================
# FLUXO 4 — Prevenção e Acompanhamento (NOVO)
# Caso A: Paciente 52 anos sem exames preventivos registrados
# Caso B: Paciente 30 anos com exames em dia
# ============================================================

print("\n")
print("=" * 60)
print("FLUXO 4: PREVENCAO E ACOMPANHAMENTO (NOVO)")
print("=" * 60)

# Caso A: varios exames em atraso
print("\n--- CASO A: Paciente 52 anos sem exames preventivos ---")
caso_prev_a = {
    "nome_anonimizado"      : "PAC_D9K3",
    "idade"                 : 52,
    "sintomas"              : ["fogachos", "cansaco"],
    "historico"             : [],                       # sem exames registrados
    "nivel_risco"           : "",
    "classificacao_urgencia": "",
    "exames_sugeridos"      : [],
    "encaminhamentos"       : [],
    "alertas"               : [],
    "orientacoes"           : [],
    "proximos_passos"       : [],
    "log_fluxo"             : [],
}

resultado_prev_a = fluxo_prevencao.invoke(caso_prev_a)
print(f"Paciente : {resultado_prev_a['nome_anonimizado']} | Idade: {resultado_prev_a['idade']} anos")
print(f"\nExames devidos identificados:")
for e in resultado_prev_a["exames_sugeridos"]:
    print(f"  - {e}")
print(f"\nAlertas preventivos:")
for a in resultado_prev_a["alertas"]:
    print(f"  *** {a}")
print(f"\nOrientacoes:")
for o in resultado_prev_a["orientacoes"]:
    print(f"  - {o}")
print(f"\nProximos Passos:")
for p in resultado_prev_a["proximos_passos"]:
    print(f"  - {p}")
print(f"\nLog do Fluxo:")
for entrada in resultado_prev_a["log_fluxo"]:
    print(f"  {entrada}")

# Caso B: exames em dia
print("\n--- CASO B: Paciente 30 anos com exames em dia ---")
caso_prev_b = {
    "nome_anonimizado"      : "PAC_E5H7",
    "idade"                 : 30,
    "sintomas"              : ["check-up rotina"],
    "historico"             : ["papanicolau realizado 2024", "vacina hpv completa"],
    "nivel_risco"           : "",
    "classificacao_urgencia": "",
    "exames_sugeridos"      : [],
    "encaminhamentos"       : [],
    "alertas"               : [],
    "orientacoes"           : [],
    "proximos_passos"       : [],
    "log_fluxo"             : [],
}

resultado_prev_b = fluxo_prevencao.invoke(caso_prev_b)
print(f"Paciente : {resultado_prev_b['nome_anonimizado']} | Idade: {resultado_prev_b['idade']} anos")
print(f"Exames devidos: {resultado_prev_b['exames_sugeridos'] or 'Nenhum — exames em dia!'}")
print(f"\nOrientacoes:")
for o in resultado_prev_b["orientacoes"]:
    print(f"  - {o}")
print(f"\nLog do Fluxo:")
for entrada in resultado_prev_b["log_fluxo"]:
    print(f"  {entrada}")

print("\n" + "=" * 60)
print("FIM DA DEMONSTRACAO DOS 4 FLUXOS")
print("=" * 60)


FLUXO 1: TRIAGEM GINECOLÓGICA

Paciente : PAC_A3F9 | Idade: 35 anos
Sintomas : dor pelvica cronica, sangramento irregular

Nivel de Risco    : ALTO
Urgencia          : LARANJA - Urgente (ate 2 horas)

Exames Sugeridos:
  - Hemograma completo
  - Ultrassom pelvico transvaginal
  - Dosagem de beta-hCG
  - PCR (Proteina C-Reativa)
  - Culturas cervicais (gonorreia/clamidia)
  - Papanicolau (se atrasado)

Orientacoes ao Profissional:
  - AVISO: Este assistente e ferramenta de apoio - validacao medica obrigatoria
  - Registrar queixas em prontuario eletronico
  - Contatar ginecologista de referencia hoje
  - Agendar consulta com prioridade

Proximos Passos:
  - Agendamento: Ate 24h - contato com ginecologista hoje
  - Enviar lembrete por SMS a paciente
  - Atualizar prontuario com resultado da triagem

Log do Fluxo (etapas com horario):
  [16:19:30] Triagem Ginecologica - risco: alto
  [16:19:30] Urgencia: LARANJA - Urgente (ate 2 horas)
  [16:19:30] 6 exames sugeridos
  [16:19:30] 4 orient